# Diagnóstico asistido por IA — IU-Xray multimodal V6.1

**Objetivo de esta etapa:** tomar la V6 como base defendible y hacer una iteración enfocada, no una reescritura completa.

V6.1 incorpora tres cambios principales:

1. Exclusión automática de nodulares con contexto benigno/calcificado no sospechoso.
2. Fusión ponderada más conservadora, con mayor peso mínimo para la imagen.
3. Selección del peso de fusión penalizando calibración deficiente mediante Brier, no solo FPR/recall.

La salida esperada al final es:

```text
/content/outputs_multimodal_iuxray_v6_1.zip
```


## 0. Guía rápida de uso en Colab

1. Sube este notebook a Colab y selecciona GPU.
2. Ejecuta todas las celdas en orden.
3. Revisa la auditoría. En V6.1 los casos nodulares benignos/calcificados no sospechosos se excluyen automáticamente del binario principal.
4. Al final descarga `outputs_multimodal_iuxray_v6_1.zip` y compártelo para analizarlo.

Configuración recomendada para la corrida principal:

```python
RUN_TRAINING_PHASE = True
MANUAL_AUDIT_APPROVED = True
RUN_KFOLD_CV = True
PRIMARY_TARGET_RECALL = 0.90
```

`PRIMARY_TARGET_RECALL = 0.90` no significa que el objetivo final sea 90% exacto en test; se usa como umbral más conservador para intentar sostener sensibilidad ≥0.85 fuera del split de threshold.


In [ ]:
# Instalación de dependencias para Colab
!pip -q install kagglehub scikit-learn tqdm matplotlib pandas numpy pillow scipy

# Backbone CXR-preentrenado opcional. Si falla, el notebook seguirá con DenseNet121 ImageNet.
!pip -q install torchxrayvision || true


## 1. Imports, configuración y reproducibilidad

Los parámetros más importantes están en esta celda. La idea es que V6.1 sea flexible sin obligarte a reescribir el notebook.

In [ ]:
import os
import re
import json
import math
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
try:
    from torchvision.models import densenet121, DenseNet121_Weights, resnet18, ResNet18_Weights, resnet50, ResNet50_Weights
    HAS_TORCHVISION_WEIGHTS = True
except Exception:
    from torchvision.models import densenet121, resnet18, resnet50
    HAS_TORCHVISION_WEIGHTS = False

try:
    import torchxrayvision as xrv
    HAS_XRV = True
except Exception as e:
    HAS_XRV = False
    xrv = None
    print("torchxrayvision no está disponible; se usará fallback si BACKBONE_NAME='xrv_densenet121'.")
    print(type(e).__name__, e)

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    brier_score_loss,
    log_loss,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

try:
    from scipy.stats import binomtest
except Exception:
    binomtest = None

import kagglehub

try:
    from IPython.display import display
except Exception:
    display = None

warnings.filterwarnings("ignore")

# --------------------------
# Configuración editable V6.1
# --------------------------
VERSION = "v6_1"
SEED = 42
KAGGLE_DATASET = "raddar/chest-xrays-indiana-university"
OUTPUT_DIR = Path("/content/outputs_multimodal_iuxray_v6_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Modo rápido para validar que todo corre. Para resultados finales: False.
QUICK_MODE = False
MAX_SAMPLES_QUICK = 800

# Bloques pesados opcionales.
RUN_KFOLD_CV = True          # V6.1 final: estimar estabilidad por UID. Si Colab se queda sin tiempo, ponlo False.
RUN_LEARNING_CURVE = False    # Para confirmar si el proyecto está data-limited, actívalo después de aprobar auditoría manual.
RUN_STACKING = True           # Secundario. La fusión ponderada es la principal.
RUN_GRADCAM = False          # Actívalo después de una corrida estable si quieres mapas de calor.

# Fase de trabajo V6.1.
# V6.1 queda lista para entrenamiento final; la auditoría estricta sigue funcionando como gate.
RUN_TRAINING_PHASE = True
MANUAL_AUDIT_APPROVED = True
REQUIRE_MANUAL_AUDIT_APPROVAL_FOR_TRAINING = True

# Auditoría estricta del target.
STRICT_LABEL_AUDIT = True
# La auditoría V6.1 separa checks críticos de checks de revisión clínica.
# Los críticos bloquean entrenamiento; los de revisión se guardan para inspección, pero no bloquean si MANUAL_AUDIT_APPROVED=True.
STRICT_LABEL_AUDIT_BLOCKS_ONLY_CRITICAL = True
MAX_ALLOWED_NORMAL_POSITIVE = 0
MAX_ALLOWED_POSITIVE_NO_INDEXING = 0
MAX_ALLOWED_POSITIVE_HYPOTHETICAL = 0
MAX_ALLOWED_POSITIVE_RESOLVED = 0
MAX_ALLOWED_ROW_PREVALENCE = 0.55
MAX_ALLOWED_UID_PREVALENCE = 0.55
MIN_NEGATIVE_ROWS_REQUIRED = 100 if not QUICK_MODE else 20
MIN_NEGATIVE_UIDS_REQUIRED = 100 if not QUICK_MODE else 20
MANUAL_REVIEW_PER_GROUP = 60

# Si problems/mesh dicen solo normal, por defecto NO dejamos que findings/impression
# lo vuelvan positivo automáticamente. Los conflictos se mandan a revisión/exclusión.
REPORT_OVERRIDE_NORMAL_TOKENS = False

# Correcciones estrictas V6.1.
# Si están activas, estos casos salen del binario principal y pasan a revisión/exclusión.
EXCLUDE_NO_INDEXING_REPORT_POSITIVES = True
EXCLUDE_HYPOTHETICAL_REPORT_POSITIVES = True
EXCLUDE_RESOLVED_REPORT_POSITIVES = True

# Cambio principal V6.1: excluir automáticamente del binario los nodulares con contexto benigno/calcificado
# sin señales sospechosas. En V6 quedaron 15 casos positivos con esta bandera; en V6.1 pasan a revisión.
EXCLUDE_BENIGN_NODULAR_CONTEXT_POSITIVES = True

# Overrides manuales opcionales: si tras revisar CSVs quieres excluir UIDs concretos, agrégalos aquí como strings.
# Ejemplo: MANUAL_EXCLUDE_UIDS = ["1234", "5678"]
MANUAL_EXCLUDE_UIDS = []

# Tarea clínica. Opciones:
# - "acute_relevant": normal vs patología aguda/relevante. Más defendible, puede dejar menos positivos.
# - "non_incidental": normal vs agudo + nodular + crónico estructural. Buen equilibrio para IU-Xray.
# - "any_abnormal": normal vs cualquier anormalidad no incidental. Más sensible, menos específico.
TASK_MODE = "non_incidental"

# Imagen.
# Recomendado: "xrv_densenet121" para usar pesos CXR-preentrenados si torchxrayvision está disponible.
# Fallbacks: "densenet121", "resnet18", "resnet50".
BACKBONE_NAME = "xrv_densenet121"
CXR_PRETRAINED_WEIGHTS = "densenet121-res224-all"
PRETRAINED = True
UNFREEZE_LAST_BLOCK = True
IMAGE_SIZE = 320              # 320 suele ser más estable en Colab. Prueba 384 si hay memoria.
BATCH_SIZE = 12
NUM_WORKERS = 2
FEATURE_DIM = 256
DROPOUT = 0.25
USE_AMP = True

# Texto. Mantener solo indication evita fuga desde findings/impression.
TEXT_FIELDS = ["indication"]
TFIDF_MAX_FEATURES = 8000
TFIDF_NGRAM_RANGE = (1, 2)

# Entrenamiento imagen.
EPOCHS_IMAGE = 10 if not QUICK_MODE else 2
LR = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3
MODEL_SELECTION = "auc_pr_mean"  # "auc_pr_mean" o "val_loss"

# Particiones por UID. Suman 1.0.
TRAIN_FIT_FRACTION = 0.60
MODEL_VAL_FRACTION = 0.10       # Early stopping / selección del checkpoint.
CALIBRATION_FRACTION = 0.10     # Platt/isotónica + peso de fusión.
THRESHOLD_FRACTION = 0.10       # Selección de umbral operativo.
TEST_FRACTION = 0.10            # Test final intocable.

# Evaluación clínica.
TARGET_RECALLS = [0.80, 0.85, 0.90]
PRIMARY_TARGET_RECALL = 0.90
THRESHOLD_GRID_SIZE = 1001
PRIMARY_IMAGE_MODEL = "image_calibrated"
PRIMARY_FUSION_MODEL = "fusion_weighted"

# Calibración.
CALIBRATION_METHOD = "platt"    # "platt" o "isotonic"
# V6.1: la rama textual fue débil en V6; por eso la fusión se vuelve conservadora.
FUSION_WEIGHT_GRID = np.round(np.linspace(0.85, 1.00, 16), 3)
FUSION_SELECTION_MODE = "fpr_brier_penalized"
FUSION_BRIER_LAMBDA = 0.50
FUSION_RECALL_SHORTFALL_PENALTY = 10.0

# Cross-validation opcional.
N_SPLITS_CV = 5
MAX_FOLDS_TO_RUN = 5 if not QUICK_MODE else 2
EPOCHS_CV = min(EPOCHS_IMAGE, 6) if not QUICK_MODE else 2

# Curva de aprendizaje opcional.
LEARNING_CURVE_FRACTIONS = [0.20, 0.40, 0.60, 0.80, 1.00] if not QUICK_MODE else [0.30, 1.00]
LEARNING_CURVE_REPEATS = 3 if not QUICK_MODE else 1
EPOCHS_LEARNING_CURVE = min(EPOCHS_IMAGE, 6) if not QUICK_MODE else 2

# Bootstrap agrupado por UID.
N_BOOTSTRAP = 2000 if not QUICK_MODE else 250
BOOTSTRAP_SEED = 123

# Otros filtros.
FRONTAL_ONLY = True
EXCLUDE_EMPTY_TEXT = False

# Ampliación opcional: CSVs propios ya armonizados con columnas mínimas uid, image_path, label.
# Por defecto queda vacío para mantener IU-Xray como benchmark base.
OPTIONAL_EXTERNAL_DATA_CSVS = []

# Reproducibilidad y dispositivo.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)
print("torchxrayvision disponible:", HAS_XRV)

if BACKBONE_NAME == "xrv_densenet121" and not HAS_XRV:
    print("BACKBONE_NAME='xrv_densenet121' solicitado, pero torchxrayvision no está disponible.")
    print("Usaré BACKBONE_NAME='densenet121' como fallback.")
    BACKBONE_NAME = "densenet121"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

config = {
    "version": VERSION,
    "seed": SEED,
    "dataset": KAGGLE_DATASET,
    "quick_mode": QUICK_MODE,
    "task_mode": TASK_MODE,
    "backbone": BACKBONE_NAME,
    "cxr_pretrained_weights": CXR_PRETRAINED_WEIGHTS if BACKBONE_NAME == "xrv_densenet121" else None,
    "pretrained": PRETRAINED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "text_fields": TEXT_FIELDS,
    "calibration_method": CALIBRATION_METHOD,
    "primary_target_recall": PRIMARY_TARGET_RECALL,
    "run_kfold_cv": RUN_KFOLD_CV,
    "run_learning_curve": RUN_LEARNING_CURVE,
    "run_training_phase": RUN_TRAINING_PHASE,
    "manual_audit_approved": MANUAL_AUDIT_APPROVED,
    "strict_label_audit": STRICT_LABEL_AUDIT,
    "max_allowed_row_prevalence": MAX_ALLOWED_ROW_PREVALENCE,
    "max_allowed_uid_prevalence": MAX_ALLOWED_UID_PREVALENCE,
    "min_negative_rows_required": MIN_NEGATIVE_ROWS_REQUIRED,
    "min_negative_uids_required": MIN_NEGATIVE_UIDS_REQUIRED,
    "report_override_normal_tokens": REPORT_OVERRIDE_NORMAL_TOKENS,
    "max_allowed_positive_no_indexing": MAX_ALLOWED_POSITIVE_NO_INDEXING,
    "max_allowed_positive_hypothetical": MAX_ALLOWED_POSITIVE_HYPOTHETICAL,
    "max_allowed_positive_resolved": MAX_ALLOWED_POSITIVE_RESOLVED,
    "exclude_no_indexing_report_positives": EXCLUDE_NO_INDEXING_REPORT_POSITIVES,
    "exclude_hypothetical_report_positives": EXCLUDE_HYPOTHETICAL_REPORT_POSITIVES,
    "exclude_resolved_report_positives": EXCLUDE_RESOLVED_REPORT_POSITIVES,
    "exclude_benign_nodular_context_positives": EXCLUDE_BENIGN_NODULAR_CONTEXT_POSITIVES,
    "manual_exclude_uids": MANUAL_EXCLUDE_UIDS,
    "fusion_weight_grid_min": float(np.min(FUSION_WEIGHT_GRID)),
    "fusion_weight_grid_max": float(np.max(FUSION_WEIGHT_GRID)),
    "fusion_selection_mode": FUSION_SELECTION_MODE,
    "fusion_brier_lambda": FUSION_BRIER_LAMBDA,
    "fusion_recall_shortfall_penalty": FUSION_RECALL_SHORTFALL_PENALTY,
    "optional_external_data_csvs": OPTIONAL_EXTERNAL_DATA_CSVS,
}

with open(OUTPUT_DIR / "config_v6_1.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))

## 2. Carga del dataset desde KaggleHub

El notebook usa el dataset de IU-Xray disponible en Kaggle. Si Colab pide autenticación, ejecuta `kagglehub.login()` en una celda aparte y vuelve a correr esta sección.

In [ ]:
print("Descargando/cargando dataset desde KaggleHub...")
try:
    dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
except Exception as e:
    print("No se pudo acceder al dataset con KaggleHub.")
    print("Si Colab solicita autenticación, ejecuta primero en una celda aparte:")
    print("kagglehub.login()")
    raise e

print("Dataset disponible en:", dataset_path)


def find_first_file(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"No se encontró {filename} dentro de {root}")
    return matches[0]

reports_csv = find_first_file(dataset_path, "indiana_reports.csv")
projections_csv = find_first_file(dataset_path, "indiana_projections.csv")

reports = pd.read_csv(reports_csv)
projections = pd.read_csv(projections_csv)

reports.columns = [c.strip().lower() for c in reports.columns]
projections.columns = [c.strip().lower() for c in projections.columns]

print("Reports CSV:", reports_csv)
print("Projections CSV:", projections_csv)
print("Columnas reports:", reports.columns.tolist())
print("Columnas projections:", projections.columns.tolist())
print("Reports shape:", reports.shape)
print("Projections shape:", projections.shape)

## 3. DataFrame multimodal y etiquetado clínico V6

Esta sección es la parte crítica de la V6.

La regla central ahora es más conservadora:

- `problems`, `mesh` y `major_mesh` siguen siendo la fuente principal de etiqueta.
- `findings` e `impression` solo apoyan la etiqueta si el hallazgo es **activo, no negado y no hipotético**.
- Casos con `label_tokens = no indexing` ya no pueden entrar como positivos por reporte narrativo; se mandan a normal si el reporte es normal, o a revisión/exclusión si hay ambigüedad.
- Hallazgos como `could obscure a small pulmonary nodule`, `possible area`, `may be`, `cannot exclude` o `interval resolution` no activan positivos en la tarea binaria principal.


In [ ]:

if "uid" not in reports.columns or "uid" not in projections.columns:
    raise ValueError("No se encontró la columna 'uid' en reports/projections.")
if "filename" not in projections.columns:
    raise ValueError("No se encontró la columna 'filename' en indiana_projections.csv.")

image_extensions = {".png", ".jpg", ".jpeg"}
image_index = {
    p.name: str(p)
    for p in dataset_path.rglob("*")
    if p.suffix.lower() in image_extensions
}

projections["image_path"] = projections["filename"].map(image_index)
missing_images = int(projections["image_path"].isna().sum())
print(f"Imágenes no encontradas por filename: {missing_images}")

df = projections.merge(reports, on="uid", how="inner")
df = df.dropna(subset=["image_path"]).copy()

if FRONTAL_ONLY and "projection" in df.columns:
    before = len(df)
    df = df[df["projection"].astype(str).str.lower().str.contains("frontal|pa|ap", na=False)].copy()
    print(f"Filtro frontal: {before} -> {len(df)} filas")
elif "projection" not in df.columns:
    print("No existe columna projection; se usará todo el conjunto disponible.")


def clean_text_value(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none", "null"}:
        return ""
    return x


def normalize_spaces(x: str) -> str:
    x = clean_text_value(x)
    x = re.sub(r"[\u2010-\u2015]", "-", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def compose_text(row, fields):
    # Texto que entra al modelo. Mantener indication evita fuga desde findings/impression.
    parts = []
    for col in fields:
        if col in row.index:
            value = normalize_spaces(row[col])
            if value:
                parts.append(value)
    text = " ".join(parts).strip()
    return text if text else "[no indication]"


def compose_label_text(row):
    # Fuente principal de etiqueta.
    candidate_cols = ["problems", "mesh", "major_mesh"]
    parts = []
    for col in candidate_cols:
        if col in row.index:
            value = normalize_spaces(row[col])
            if value:
                parts.append(value)
    return " ; ".join(parts)


def compose_report_text(row):
    # Fuente narrativa auxiliar, con negation/uncertainty handling.
    candidate_cols = ["findings", "impression"]
    parts = []
    for col in candidate_cols:
        if col in row.index:
            value = normalize_spaces(row[col])
            if value:
                parts.append(value)
    return " ; ".join(parts)


def compose_evidence_text(row):
    parts = []
    for col in ["problems", "mesh", "major_mesh", "findings", "impression"]:
        if col in row.index:
            value = normalize_spaces(row[col])
            if value:
                parts.append(value)
    return " ; ".join(parts)


def split_label_tokens(text):
    text = clean_text_value(text).lower()
    if not text:
        return []
    raw = re.split(r"[;,|]", text)
    tokens = []
    seen = set()
    for t in raw:
        t = re.sub(r"\s+", " ", t).strip()
        if t and t not in {"nan", "none", "null"} and t not in seen:
            tokens.append(t)
            seen.add(t)
    return tokens


def contains_pattern(text, patterns):
    text = clean_text_value(text).lower()
    return any(re.search(p, text) for p in patterns)


def sentence_split(text):
    text = normalize_spaces(text).lower()
    if not text:
        return []
    chunks = re.split(r"(?<=[\.!?])\s+|;|\n", text)
    return [c.strip(" .") for c in chunks if c.strip(" .")]

# -------------------------------
# Patrones clínicos V6
# -------------------------------
STRONG_ACUTE_PATTERNS = [
    r"\bpneumonia\b",
    r"consolidat",
    r"pneumothorax",
    r"pleural effusion",
    r"\beffusion\b",
    r"pulmonary edema",
    r"\bedema\b",
    r"pulmonary venous hypertension",
    r"vascular congestion",
    r"\bcongestion\b",
    r"infiltrat",
    r"airspace disease",
    r"air space disease",
    r"opacification",
    r"pneumonitis",
]

ATELECTASIS_OPACITY_PATTERNS = [
    r"atelect",
    r"\bopacity\b",
    r"opacities",
]

MILD_QUALIFIER_PATTERNS = [
    r"minimal",
    r"mild",
    r"linear",
    r"streaky",
    r"subsegmental",
    r"bibasal",
    r"bibasilar",
    r"low volume",
    r"low lung volume",
    r"hypoinflat",
    r"basilar atelectatic",
    r"plate[- ]like",
    r"subtle",
    r"borderline",
    r"trace",
    r"tiny",
]

NODULAR_PATTERNS = [
    r"lung nodule",
    r"pulmonary nodule",
    r"\bnodule\b",
    r"lung mass",
    r"pulmonary mass",
    r"\bmass\b",
]

# Refinamiento V6.1: no todo hallazgo nodular debe entrar como positivo si el contexto
# apunta a granuloma/nódulo calcificado benigno. Una masa o nódulo no calcificado
# sigue tratándose como positivo.
BENIGN_NODULAR_CONTEXT_PATTERNS = [
    r"calcified granuloma",
    r"calcified pulmonary granuloma",
    r"calcified lung granuloma",
    r"calcified nodule",
    r"calcified pulmonary nodule",
    r"calcified lung nodule",
    r"granuloma",
    r"granulomatous",
]

SUSPICIOUS_NODULAR_CONTEXT_PATTERNS = [
    r"non[- ]?calcified.*nodule",
    r"new.*nodule",
    r"enlarging.*nodule",
    r"spiculated",
    r"lung mass",
    r"pulmonary mass",
    r"\bmass\b",
    r"recommend.*(ct|computed tomography)",
    r"(ct|computed tomography).*recommend",
    r"follow[- ]?up.*(ct|computed tomography|nodule|mass)",
]

CHRONIC_STRUCTURAL_PATTERNS = [
    r"cardiomegal",
    r"heart enlargement",
    r"enlarged cardiac silhouette",
    r"emphysema",
    r"hyperinflation",
    r"\bcopd\b",
    r"fibrosis",
    r"interstitial",
    r"chronic interstitial",
]

INCIDENTAL_EXCLUSION_PATTERNS = [
    r"calcified granuloma",
    r"granuloma",
    r"granulomatous",
    r"cicatrix",
    r"scarring",
    r"\bscar\b",
    r"hiatal hernia",
    r"hernia, hiatal",
    r"tortuous aorta",
    r"aortic atherosclerosis",
    r"aortic calcification",
    r"aorta, thoracic",
    r"spine",
    r"thoracic vertebrae",
    r"vertebra",
    r"scoliosis",
    r"kyphosis",
    r"osteophyte",
    r"spondylosis",
    r"degenerative",
    r"bone diseases",
    r"surgical instruments",
    r"catheter",
    r"catheters",
    r"medical device",
    r"implanted medical device",
    r"pacemaker",
    r"prostheses",
    r"technical quality",
    r"quality of image unsatisfactory",
    r"foreign bodies",
    r"rib fracture",
    r"\bfracture\b",
]

NORMAL_TOKEN_PATTERNS = [
    r"^normal$",
    r"^no finding[s]?$",
    r"^negative$",
    r"^unremarkable$",
]

NO_INDEXING_TOKEN_PATTERNS = [
    r"^no indexing$",
    r"^no index$",
    r"^no indexing terms?$",
]

NORMAL_REPORT_PATTERNS = [
    r"no acute cardio\s?pulmonary (disease|abnormality|process|findings?)",
    r"no active cardio\s?pulmonary disease",
    r"no acute pulmonary (disease|abnormality|process|findings?)",
    r"no acute disease",
    r"no acute findings?",
    r"no acute abnormality",
    r"lungs (are|appear) clear",
    r"the lungs (are|appear) clear",
    r"clear lungs",
    r"pleural spaces are clear",
    r"heart size is normal",
    r"cardiomediastinal silhouette is normal",
    r"no focal (airspace )?(opacity|consolidation|infiltrate)",
    r"no evidence of pneumonia",
    r"no pneumonia",
]

NEGATION_CUES = [
    r"\bno\b",
    r"\bnot\b",
    r"without",
    r"absence of",
    r"absent",
    r"negative for",
    r"no evidence of",
    r"no definite",
    r"no focal",
    r"no acute",
    r"free of",
    r"clear of",
]

POST_NEGATION_CUES = [
    r"not seen",
    r"not present",
    r"is absent",
    r"are absent",
    r"is not identified",
    r"are not identified",
    r"is not visualized",
    r"are not visualized",
]

UNCERTAINTY_OR_RULEOUT_CUES = [
    r"rule out",
    r"r/o",
    r"question of",
    r"\?",
    r"possible",
    r"possibly",
    r"may",
    r"could",
    r"questionable",
    r"equivocal",
    r"suspected",
    r"suspicious for",
    r"cannot exclude",
    r"not excluded",
    r"may represent",
    r"could represent",
    r"if clinically indicated",
]

HISTORICAL_NONACTIVE_CUES = [
    r"history of",
    r"prior",
    r"remote",
    r"old",
    r"resolved",
    r"resolution",
    r"resolving",
    r"interval resolution",
    r"improved",
]

HYPOTHETICAL_POSITIVE_PATTERNS = [
    r"could obscure.*(nodule|mass|opacity|lesion)",
    r"(ct|computed tomography).*more sensitive.*(nodule|lesion|mass)",
    r"if clinically indicated.*(nodule|lesion|mass|opacity)",
    r"there may be.*(opacity|infiltrate|consolidation|pneumonitis|nodule|mass)",
    r"may be.*(opacity|infiltrate|consolidation|pneumonitis|nodule|mass)",
    r"possible.*(opacity|infiltrate|consolidation|pneumonitis|nodule|mass|effusion)",
    r"questionable.*(opacity|infiltrate|consolidation|pneumonitis|nodule|mass|effusion)",
    r"cannot exclude.*(opacity|infiltrate|consolidation|pneumonitis|nodule|mass|effusion)",
    r"not excluded.*(opacity|infiltrate|consolidation|pneumonitis|nodule|mass|effusion)",
]

RESOLVED_POSITIVE_PATTERNS = [
    r"interval resolution of.*(pleural effusion|effusion|pneumothorax|edema|consolidation|infiltrate|opacity|pneumonia|airspace)",
    r"resol(?:ved|ution|ving).*(pleural effusion|effusion|pneumothorax|edema|consolidation|infiltrate|opacity|pneumonia|airspace)",
    r"(pleural effusion|effusion|pneumothorax|edema|consolidation|infiltrate|opacity|pneumonia|airspace).*resol(?:ved|ution|ving)",
    r"(previous|prior|old).*(pleural effusion|effusion|pneumothorax|edema|consolidation|infiltrate|opacity|pneumonia|airspace).*(resolved|resolution|improved)",
]

ALL_TARGET_POSITIVE_PATTERNS = (
    STRONG_ACUTE_PATTERNS
    + ATELECTASIS_OPACITY_PATTERNS
    + NODULAR_PATTERNS
    + CHRONIC_STRUCTURAL_PATTERNS
)


def token_is_normal(token):
    token = normalize_spaces(token).lower()
    return any(re.search(p, token) for p in NORMAL_TOKEN_PATTERNS)


def token_is_no_indexing(token):
    token = normalize_spaces(token).lower()
    return any(re.search(p, token) for p in NO_INDEXING_TOKEN_PATTERNS)


def tokens_are_only_normal(tokens):
    tokens = [normalize_spaces(t).lower() for t in tokens if normalize_spaces(t)]
    return len(tokens) > 0 and all(token_is_normal(t) for t in tokens)


def tokens_have_no_indexing(tokens):
    tokens = [normalize_spaces(t).lower() for t in tokens if normalize_spaces(t)]
    return any(token_is_no_indexing(t) for t in tokens)


def tokens_are_only_no_indexing(tokens):
    tokens = [normalize_spaces(t).lower() for t in tokens if normalize_spaces(t)]
    return len(tokens) > 0 and all(token_is_no_indexing(t) for t in tokens)


def is_negated_or_nonactive(sentence, start, end, window_before=12, window_after=8):
    sentence = sentence.lower()
    prefix_words = re.findall(r"\b\w+[\w-]*\b", sentence[:start])
    suffix_words = re.findall(r"\b\w+[\w-]*\b", sentence[end:])
    before = " ".join(prefix_words[-window_before:])
    after = " ".join(suffix_words[:window_after])

    if any(re.search(p, before) for p in NEGATION_CUES):
        return True
    if any(re.search(p, after) for p in POST_NEGATION_CUES):
        return True
    if any(re.search(p, before) for p in UNCERTAINTY_OR_RULEOUT_CUES):
        return True
    if any(re.search(p, before) for p in HISTORICAL_NONACTIVE_CUES):
        return True
    return False


def find_unnegated_matches(text, patterns):
    matches = []
    for sentence in sentence_split(text):
        for pattern in patterns:
            for match in re.finditer(pattern, sentence):
                if not is_negated_or_nonactive(sentence, match.start(), match.end()):
                    matches.append({
                        "pattern": pattern,
                        "match": match.group(0),
                        "sentence": sentence,
                    })
    return matches


def has_unnegated_pattern(text, patterns):
    return len(find_unnegated_matches(text, patterns)) > 0


def report_has_hypothetical_positive(text):
    text = clean_text_value(text).lower()
    if contains_pattern(text, HYPOTHETICAL_POSITIVE_PATTERNS):
        return True
    for sentence in sentence_split(text):
        if contains_pattern(sentence, UNCERTAINTY_OR_RULEOUT_CUES) and contains_pattern(sentence, ALL_TARGET_POSITIVE_PATTERNS):
            return True
    return False


def report_has_resolved_positive(text):
    text = clean_text_value(text).lower()
    if contains_pattern(text, RESOLVED_POSITIVE_PATTERNS):
        return True
    for sentence in sentence_split(text):
        if contains_pattern(sentence, HISTORICAL_NONACTIVE_CUES) and contains_pattern(sentence, ALL_TARGET_POSITIVE_PATTERNS):
            return True
    return False


def classify_structured_tokens(tokens):
    token_text = " ; ".join(tokens).lower()
    if not token_text:
        return None, {}

    only_normal = tokens_are_only_normal(tokens)
    only_no_indexing = tokens_are_only_no_indexing(tokens)
    has_no_indexing = tokens_have_no_indexing(tokens)
    flags = {
        "structured_only_normal": only_normal,
        "structured_has_no_indexing": has_no_indexing,
        "structured_only_no_indexing": only_no_indexing,
        "structured_has_strong_acute": contains_pattern(token_text, STRONG_ACUTE_PATTERNS),
        "structured_has_atelectasis_or_opacity": contains_pattern(token_text, ATELECTASIS_OPACITY_PATTERNS),
        "structured_has_mild_qualifier": contains_pattern(token_text, MILD_QUALIFIER_PATTERNS),
        "structured_has_nodular": contains_pattern(token_text, NODULAR_PATTERNS),
        "structured_has_benign_nodular_context": contains_pattern(token_text, BENIGN_NODULAR_CONTEXT_PATTERNS),
        "structured_has_suspicious_nodular_context": contains_pattern(token_text, SUSPICIOUS_NODULAR_CONTEXT_PATTERNS),
        "structured_has_chronic_structural": contains_pattern(token_text, CHRONIC_STRUCTURAL_PATTERNS),
        "structured_has_incidental": contains_pattern(token_text, INCIDENTAL_EXCLUSION_PATTERNS),
        "structured_has_normal_token": any(token_is_normal(t) for t in tokens),
    }

    # no indexing no es evidencia positiva ni normal por sí mismo.
    if only_no_indexing:
        return None, flags

    if flags["structured_has_strong_acute"]:
        return "acute_relevant", flags
    if flags["structured_has_atelectasis_or_opacity"]:
        if flags["structured_has_mild_qualifier"] and not flags["structured_has_nodular"]:
            return "mild_ambiguous", flags
        return "acute_relevant", flags
    if flags["structured_has_nodular"]:
        if flags["structured_has_benign_nodular_context"] and not flags["structured_has_suspicious_nodular_context"]:
            return "incidental_excluded", flags
        return "nodular", flags
    if flags["structured_has_chronic_structural"]:
        return "chronic_structural", flags
    if flags["structured_has_incidental"]:
        return "incidental_excluded", flags
    if only_normal:
        return "normal", flags
    return None, flags


def classify_report_text(report_text):
    report_text = clean_text_value(report_text).lower()
    hypothetical = report_has_hypothetical_positive(report_text)
    resolved = report_has_resolved_positive(report_text)
    flags = {
        "report_has_hypothetical_positive": hypothetical,
        "report_has_resolved_positive": resolved,
        "report_has_strong_acute_unnegated": has_unnegated_pattern(report_text, STRONG_ACUTE_PATTERNS),
        "report_has_atelectasis_or_opacity_unnegated": has_unnegated_pattern(report_text, ATELECTASIS_OPACITY_PATTERNS),
        "report_has_nodular_unnegated": has_unnegated_pattern(report_text, NODULAR_PATTERNS),
        "report_has_benign_nodular_context": contains_pattern(report_text, BENIGN_NODULAR_CONTEXT_PATTERNS),
        "report_has_suspicious_nodular_context": contains_pattern(report_text, SUSPICIOUS_NODULAR_CONTEXT_PATTERNS),
        "report_has_chronic_structural_unnegated": has_unnegated_pattern(report_text, CHRONIC_STRUCTURAL_PATTERNS),
        "report_has_incidental_unnegated": has_unnegated_pattern(report_text, INCIDENTAL_EXCLUSION_PATTERNS),
        "report_has_mild_qualifier": contains_pattern(report_text, MILD_QUALIFIER_PATTERNS),
        "report_has_normal_phrase": contains_pattern(report_text, NORMAL_REPORT_PATTERNS),
    }

    # Primero sacamos hallazgos explícitamente no activos/hipotéticos.
    if resolved:
        return "resolved_nonactive_excluded", flags
    if hypothetical:
        return "hypothetical_unclear_excluded", flags

    if flags["report_has_strong_acute_unnegated"]:
        return "acute_relevant", flags
    if flags["report_has_atelectasis_or_opacity_unnegated"]:
        if flags["report_has_mild_qualifier"] and not flags["report_has_nodular_unnegated"]:
            return "mild_ambiguous", flags
        return "acute_relevant", flags
    if flags["report_has_nodular_unnegated"]:
        if flags["report_has_benign_nodular_context"] and not flags["report_has_suspicious_nodular_context"]:
            return "incidental_excluded", flags
        return "nodular", flags
    if flags["report_has_chronic_structural_unnegated"]:
        return "chronic_structural", flags
    if flags["report_has_incidental_unnegated"]:
        return "incidental_excluded", flags
    if flags["report_has_normal_phrase"]:
        return "normal", flags
    return None, flags


def is_positive_category(category):
    return category in {"acute_relevant", "nodular", "chronic_structural"}


def derive_v6_1_category(row):
    label_text = compose_label_text(row)
    report_text = compose_report_text(row)
    tokens = split_label_tokens(label_text)

    structured_category, structured_flags = classify_structured_tokens(tokens)
    report_category, report_flags = classify_report_text(report_text)

    structured_only_normal = structured_flags.get("structured_only_normal", False)
    structured_has_no_indexing = structured_flags.get("structured_has_no_indexing", False)
    structured_only_no_indexing = structured_flags.get("structured_only_no_indexing", False)
    report_hypothetical = report_flags.get("report_has_hypothetical_positive", False)
    report_resolved = report_flags.get("report_has_resolved_positive", False)
    report_normal = report_flags.get("report_has_normal_phrase", False)

    # Corrección 1: si los campos estructurados dicen SOLO normal, no convertimos a positivo por reporte narrativo.
    if structured_only_normal:
        if is_positive_category(report_category):
            if REPORT_OVERRIDE_NORMAL_TOKENS:
                return report_category
            return "discordant_review_excluded"
        if report_category in {"hypothetical_unclear_excluded", "resolved_nonactive_excluded"}:
            return report_category
        return "normal"

    # Corrección 2: no indexing no debe producir positivos en el binario principal.
    # Primero preservamos las exclusiones por hallazgos hipotéticos/resueltos; después, si el reporte es normal, queda como normal.
    if structured_only_no_indexing or (structured_has_no_indexing and EXCLUDE_NO_INDEXING_REPORT_POSITIVES):
        if report_category == "resolved_nonactive_excluded":
            return "resolved_nonactive_excluded"
        if report_category == "hypothetical_unclear_excluded":
            return "hypothetical_unclear_excluded"
        if report_category == "normal":
            return "normal"
        if report_normal and not is_positive_category(report_category):
            return "normal"
        if is_positive_category(report_category):
            return "no_indexing_report_review_excluded"
        return "no_indexing_unclear_excluded"

    # Corrección 3: hallazgos resueltos/hipotéticos hacen que el caso salga del binario principal.
    if EXCLUDE_RESOLVED_REPORT_POSITIVES and report_resolved:
        return "resolved_nonactive_excluded"
    if EXCLUDE_HYPOTHETICAL_REPORT_POSITIVES and report_hypothetical:
        return "hypothetical_unclear_excluded"

    # Si los campos estructurados ya contienen una categoría clara, se respeta.
    if structured_category is not None:
        return structured_category

    # Si no hay categoría estructurada clara, se usa el reporte narrativo ya filtrado.
    if report_category is not None:
        return report_category

    return "unclear_excluded"


def map_category_to_binary(category, task_mode):
    if task_mode == "acute_relevant":
        positive = {"acute_relevant"}
        negative = {"normal"}
    elif task_mode == "non_incidental":
        positive = {"acute_relevant", "nodular", "chronic_structural"}
        negative = {"normal"}
    elif task_mode == "any_abnormal":
        positive = {"acute_relevant", "nodular", "chronic_structural", "mild_ambiguous"}
        negative = {"normal"}
    else:
        raise ValueError("TASK_MODE inválido. Usa: acute_relevant, non_incidental o any_abnormal.")

    if category in positive:
        return 1
    if category in negative:
        return 0
    return np.nan

# Construcción final.
df["text"] = df.apply(lambda row: compose_text(row, TEXT_FIELDS), axis=1)
df["label_text"] = df.apply(compose_label_text, axis=1)
df["report_text"] = df.apply(compose_report_text, axis=1)
df["evidence_text"] = df.apply(compose_evidence_text, axis=1)
df["label_tokens_list"] = df["label_text"].apply(split_label_tokens)
df["label_tokens"] = df["label_tokens_list"].apply(lambda xs: " | ".join(xs))
df["structured_only_normal"] = df["label_tokens_list"].apply(tokens_are_only_normal)
df["structured_has_no_indexing"] = df["label_tokens_list"].apply(tokens_have_no_indexing)
df["structured_only_no_indexing"] = df["label_tokens_list"].apply(tokens_are_only_no_indexing)
df["report_has_hypothetical_positive"] = df["report_text"].apply(report_has_hypothetical_positive)
df["report_has_resolved_positive"] = df["report_text"].apply(report_has_resolved_positive)
df["evidence_has_benign_nodular_context"] = df["evidence_text"].apply(lambda x: contains_pattern(x, BENIGN_NODULAR_CONTEXT_PATTERNS))
df["evidence_has_suspicious_nodular_context"] = df["evidence_text"].apply(lambda x: contains_pattern(x, SUSPICIOUS_NODULAR_CONTEXT_PATTERNS))
df["diagnostic_category"] = df.apply(derive_v6_1_category, axis=1)

# V6.1: exclusión explícita de nodulares benignos/calcificados no sospechosos.
benign_nodular_review_mask = (
    df["diagnostic_category"].eq("nodular") &
    df["evidence_has_benign_nodular_context"].eq(True) &
    ~df["evidence_has_suspicious_nodular_context"].eq(True)
)
if EXCLUDE_BENIGN_NODULAR_CONTEXT_POSITIVES:
    print("V6.1: nodulares benignos/calcificados movidos a revisión/exclusión:", int(benign_nodular_review_mask.sum()))
    df.loc[benign_nodular_review_mask, "diagnostic_category"] = "benign_nodular_review_excluded"

# V6.1: exclusión manual opcional por UID, si se llena MANUAL_EXCLUDE_UIDS en la configuración.
if MANUAL_EXCLUDE_UIDS:
    manual_exclude_set = {str(x) for x in MANUAL_EXCLUDE_UIDS}
    manual_exclude_mask = df["uid"].astype(str).isin(manual_exclude_set)
    print("V6.1: filas excluidas manualmente por UID:", int(manual_exclude_mask.sum()))
    df.loc[manual_exclude_mask, "diagnostic_category"] = "manual_review_excluded"

df["label"] = df["diagnostic_category"].apply(lambda c: map_category_to_binary(c, TASK_MODE))
df["row_id"] = np.arange(len(df))

print("Distribución de categorías V6.1 antes de filtrar:")
category_counts = df["diagnostic_category"].value_counts(dropna=False).reset_index()
category_counts.columns = ["diagnostic_category", "n_rows"]
display(category_counts)
category_counts.to_csv(OUTPUT_DIR / "category_distribution_before_filter_v6_1.csv", index=False)

used_df = df.dropna(subset=["label"]).copy()
used_df["label"] = used_df["label"].astype(int)

if EXCLUDE_EMPTY_TEXT:
    before = len(used_df)
    used_df = used_df[used_df["text"].astype(str).str.strip().ne("[no indication]")].copy()
    print(f"Filtro texto vacío: {before} -> {len(used_df)} filas")

if QUICK_MODE:
    # Submuestreo por UID para que siga sin fuga.
    uid_table_quick = used_df.groupby("uid")["label"].max().reset_index()
    sample_n = min(MAX_SAMPLES_QUICK, len(uid_table_quick))
    if sample_n >= len(uid_table_quick):
        sampled_uids = uid_table_quick.copy()
    else:
        vc = uid_table_quick["label"].value_counts()
        strat = uid_table_quick["label"] if len(vc) >= 2 and vc.min() >= 2 else None
        sampled_uids, _ = train_test_split(
            uid_table_quick,
            train_size=sample_n,
            random_state=SEED,
            stratify=strat,
        )
    used_df = used_df[used_df["uid"].isin(set(sampled_uids["uid"]))].copy()
    print(f"QUICK_MODE: usando {len(used_df)} filas de {sample_n} UIDs aproximados.")

used_df = used_df.reset_index(drop=True)

print("\nDataset usado para la tarea:", TASK_MODE)
print("Filas:", len(used_df), "UIDs:", used_df["uid"].nunique())
print("Prevalencia por fila:", used_df["label"].mean())
print("Prevalencia por UID:", used_df.groupby("uid")["label"].max().mean())

summary_used = pd.DataFrame({
    "n_rows": [len(used_df)],
    "n_uids": [used_df["uid"].nunique()],
    "row_prevalence": [used_df["label"].mean()],
    "uid_prevalence": [used_df.groupby("uid")["label"].max().mean()],
    "task_mode": [TASK_MODE],
})
summary_used.to_csv(OUTPUT_DIR / "dataset_summary_v6_1.csv", index=False)

df.to_csv(OUTPUT_DIR / "dataset_labeled_full_v6_1.csv", index=False)
used_df.to_csv(OUTPUT_DIR / "dataset_used_binary_v6_1.csv", index=False)
excluded_df = df[df["label"].isna()].copy()
excluded_df.to_csv(OUTPUT_DIR / "dataset_excluded_v6_1.csv", index=False)

cols_preview = [c for c in [
    "uid", "filename", "projection", "diagnostic_category", "label", "structured_only_normal",
    "structured_has_no_indexing", "structured_only_no_indexing", "report_has_hypothetical_positive",
    "report_has_resolved_positive", "evidence_has_benign_nodular_context",
    "evidence_has_suspicious_nodular_context", "label_tokens", "text", "report_text"
] if c in used_df.columns]
display(used_df[cols_preview].head(12))


## 3.1. Ampliación opcional con datos externos compatibles

Esta celda permite sumar datos ya etiquetados desde CSVs propios, por ejemplo de MIMIC-CXR-JPG o CheXpert si ya tienes acceso y una etiqueta armonizada. Por defecto no hace nada porque `OPTIONAL_EXTERNAL_DATA_CSVS=[]`.

In [ ]:

def load_optional_external_csvs(csv_paths):
    external_frames = []
    for csv_path in csv_paths:
        csv_path = Path(csv_path)
        if not csv_path.exists():
            print(f"CSV externo no encontrado, se omite: {csv_path}")
            continue
        ext = pd.read_csv(csv_path)
        ext.columns = [c.strip().lower() for c in ext.columns]
        required = {"uid", "image_path", "label"}
        missing = required - set(ext.columns)
        if missing:
            raise ValueError(f"{csv_path} no tiene columnas requeridas: {missing}. Requeridas: uid, image_path, label")
        ext = ext.copy()
        source_name = csv_path.stem
        ext["uid"] = "external_" + source_name + "_" + ext["uid"].astype(str)
        ext["image_path"] = ext["image_path"].astype(str)
        ext = ext[ext["image_path"].apply(lambda p: Path(p).exists())].copy()
        ext["filename"] = ext["image_path"].apply(lambda p: Path(p).name)
        ext["projection"] = ext.get("projection", "external")
        if "text" not in ext.columns:
            if "indication" in ext.columns:
                ext["text"] = ext["indication"].fillna("missing indication").astype(str)
            else:
                ext["text"] = "missing indication"
        ext["label"] = ext["label"].astype(int)
        ext["label_text"] = ext.get("label_text", "external_label")
        ext["label_tokens"] = ext.get("label_tokens", "external_label")
        ext["diagnostic_category_external"] = np.where(ext["label"] == 1, "external_positive", "external_negative")
        ext["diagnostic_category"] = ext["diagnostic_category_external"]
        for col in ["has_acute_relevant", "has_structural_chronic", "has_nodular", "has_incidental_excluded", "has_normal_token"]:
            if col not in ext.columns:
                ext[col] = np.nan
        external_frames.append(ext)
        print(f"Datos externos añadibles desde {csv_path}: {len(ext)} filas válidas")
    return external_frames

external_frames = load_optional_external_csvs(OPTIONAL_EXTERNAL_DATA_CSVS)
if external_frames:
    base_cols = list(used_df.columns)
    aligned_frames = [used_df]
    for ext in external_frames:
        for col in base_cols:
            if col not in ext.columns:
                ext[col] = np.nan
        aligned_frames.append(ext[base_cols])
    used_df = pd.concat(aligned_frames, ignore_index=True)
    used_df["row_id"] = np.arange(len(used_df))
    print("Dataset tras añadir externos:", used_df.shape)
    print("Prevalencia combinada:", used_df["label"].mean())
    used_df.to_csv(OUTPUT_DIR / "dataset_labeled_plus_optional_external_v6_1.csv", index=False)
else:
    print("No se añadieron datos externos. IU-Xray queda como benchmark base.")


## 3.2. Auditoría obligatoria de etiquetas V6.1 — corregida

Antes de entrenar, esta celda revisa que el target no esté roto. En esta versión corregida la auditoría separa dos niveles:

- **Crítico/bloqueante**: errores que no deberían pasar bajo ninguna circunstancia, como positivos con tokens estructurados solo normales, positivos con `no indexing`, prevalencia anormalmente alta o falta de clases suficientes.
- **Revisión clínica/no bloqueante**: casos que deben quedar guardados para inspección, como positivos con frases hipotéticas/resueltas o contexto nodular benigno/calcificado. Estos patrones son deliberadamente amplios y pueden aparecer en reportes mixtos; por eso no bloquean todo el entrenamiento si la revisión manual ya fue aprobada.

Archivos clave:

- `label_audit_summary_v6_1.csv`
- `label_audit_suspect_cases_v6_1.csv`
- `manual_review_sample_v6_1.csv`
- `edge_cases_to_review_v6_1.csv`

La lógica recomendada es: si falla `AUDIT_GATE_PASS`, detenerse; si solo falla `AUDIT_PASS_ALL`, revisar los sospechosos pero permitir entrenamiento cuando `MANUAL_AUDIT_APPROVED=True`.


In [ ]:

def first_existing_cols(frame, cols):
    return [c for c in cols if c in frame.columns]

# 1) Inconsistencia crítica: tokens estructurados solo normales pero label positivo.
bad_normal_positive = used_df[
    (used_df["label"] == 1) &
    (used_df["structured_only_normal"] == True)
].copy()

# 2) Casos no indexing positivos: no se permiten en V6.
positive_no_indexing = used_df[
    (used_df["label"] == 1) &
    (used_df["structured_has_no_indexing"] == True)
].copy()

# 3) Positivos con reporte hipotético/resuelto: tampoco se permiten en el binario principal.
positive_hypothetical = used_df[
    (used_df["label"] == 1) &
    (used_df["report_has_hypothetical_positive"] == True)
].copy()
positive_resolved = used_df[
    (used_df["label"] == 1) &
    (used_df["report_has_resolved_positive"] == True)
].copy()

# 3b) V6.1: positivos nodulares con contexto benigno/calcificado sin señal sospechosa.
positive_benign_nodular_context = used_df[
    (used_df["label"] == 1) &
    (used_df["diagnostic_category"].eq("nodular")) &
    (used_df.get("evidence_has_benign_nodular_context", False) == True) &
    (used_df.get("evidence_has_suspicious_nodular_context", False) == False)
].copy()

# 4) Casos excluidos por las reglas nuevas.
review_categories = {
    "discordant_review_excluded",
    "no_indexing_report_review_excluded",
    "no_indexing_unclear_excluded",
    "hypothetical_unclear_excluded",
    "resolved_nonactive_excluded",
    "benign_nodular_review_excluded",
    "manual_review_excluded",
}
edge_cases = df[df["diagnostic_category"].astype(str).isin(review_categories)].copy()
benign_nodular_review_candidates = df[df["diagnostic_category"].eq("benign_nodular_review_excluded")].copy()

# 5) Prevalencia y tamaño negativo.
row_prevalence = float(used_df["label"].mean()) if len(used_df) else np.nan
uid_prevalence = float(used_df.groupby("uid")["label"].max().mean()) if len(used_df) else np.nan
negative_rows = int((used_df["label"] == 0).sum())
negative_uids = int((used_df.groupby("uid")["label"].max() == 0).sum())
positive_rows = int((used_df["label"] == 1).sum())
positive_uids = int((used_df.groupby("uid")["label"].max() == 1).sum())

# 6) Frases negadas/hipotéticas/resueltas frecuentes.
phrase_patterns = [
    ("negated_no_acute_cardiopulmonary", r"no acute cardio\s?pulmonary"),
    ("negated_no_pneumothorax", r"no pneumothorax"),
    ("negated_no_pleural_effusion", r"no pleural effusion"),
    ("negated_no_focal_consolidation", r"no focal consolidation"),
    ("negated_no_focal_infiltrate", r"no focal infiltrate"),
    ("negated_no_evidence_pneumonia", r"no evidence of pneumonia"),
    ("hypothetical_could_obscure_nodule", r"could obscure.*nodule"),
    ("hypothetical_possible", r"possible"),
    ("hypothetical_may_be", r"may be"),
    ("resolved_interval_resolution", r"interval resolution"),
    ("benign_calcified_granuloma", r"calcified granuloma|calcified.*nodule|granuloma"),
]
phrase_rows = []
for name, pattern in phrase_patterns:
    all_mask = df["report_text"].fillna("").str.lower().str.contains(pattern, regex=True).fillna(False)
    used_pos_mask = used_df["report_text"].fillna("").str.lower().str.contains(pattern, regex=True).fillna(False) & used_df["label"].eq(1)
    phrase_rows.append({"pattern_name": name, "pattern": pattern, "n_all": int(all_mask.sum()), "n_positive_used": int(used_pos_mask.sum())})
phrase_counts_df = pd.DataFrame(phrase_rows)

# Resumen de auditoría.
# V6.1: se separan checks críticos/bloqueantes de checks de revisión clínica.
# Los checks de revisión son deliberadamente sensibles; pueden capturar reportes mixtos y por eso no bloquean
# el entrenamiento cuando la auditoría manual ya fue aprobada.
audit_rows = [
    {
        "check": "normal_tokens_labeled_positive",
        "severity": "critical",
        "value": len(bad_normal_positive),
        "limit": MAX_ALLOWED_NORMAL_POSITIVE,
        "pass": len(bad_normal_positive) <= MAX_ALLOWED_NORMAL_POSITIVE,
        "action": "bloquear_si_falla",
    },
    {
        "check": "positive_no_indexing",
        "severity": "critical",
        "value": len(positive_no_indexing),
        "limit": MAX_ALLOWED_POSITIVE_NO_INDEXING,
        "pass": len(positive_no_indexing) <= MAX_ALLOWED_POSITIVE_NO_INDEXING,
        "action": "bloquear_si_falla",
    },
    {
        "check": "positive_hypothetical",
        "severity": "review",
        "value": len(positive_hypothetical),
        "limit": MAX_ALLOWED_POSITIVE_HYPOTHETICAL,
        "pass": len(positive_hypothetical) <= MAX_ALLOWED_POSITIVE_HYPOTHETICAL,
        "action": "revisar_sospechosos_no_bloquear",
    },
    {
        "check": "positive_resolved",
        "severity": "review",
        "value": len(positive_resolved),
        "limit": MAX_ALLOWED_POSITIVE_RESOLVED,
        "pass": len(positive_resolved) <= MAX_ALLOWED_POSITIVE_RESOLVED,
        "action": "revisar_sospechosos_no_bloquear",
    },
    {
        "check": "positive_benign_nodular_context",
        "severity": "review",
        "value": len(positive_benign_nodular_context),
        "limit": 0,
        "pass": len(positive_benign_nodular_context) == 0,
        "action": "revisar_sospechosos_no_bloquear",
    },
    {
        "check": "row_prevalence",
        "severity": "critical",
        "value": row_prevalence,
        "limit": MAX_ALLOWED_ROW_PREVALENCE,
        "pass": row_prevalence <= MAX_ALLOWED_ROW_PREVALENCE,
        "action": "bloquear_si_falla",
    },
    {
        "check": "uid_prevalence",
        "severity": "critical",
        "value": uid_prevalence,
        "limit": MAX_ALLOWED_UID_PREVALENCE,
        "pass": uid_prevalence <= MAX_ALLOWED_UID_PREVALENCE,
        "action": "bloquear_si_falla",
    },
    {
        "check": "negative_rows",
        "severity": "critical",
        "value": negative_rows,
        "limit": MIN_NEGATIVE_ROWS_REQUIRED,
        "pass": negative_rows >= MIN_NEGATIVE_ROWS_REQUIRED,
        "action": "bloquear_si_falla",
    },
    {
        "check": "negative_uids",
        "severity": "critical",
        "value": negative_uids,
        "limit": MIN_NEGATIVE_UIDS_REQUIRED,
        "pass": negative_uids >= MIN_NEGATIVE_UIDS_REQUIRED,
        "action": "bloquear_si_falla",
    },
    {
        "check": "positive_rows",
        "severity": "critical",
        "value": positive_rows,
        "limit": 1,
        "pass": positive_rows >= 1,
        "action": "bloquear_si_falla",
    },
    {
        "check": "positive_uids",
        "severity": "critical",
        "value": positive_uids,
        "limit": 1,
        "pass": positive_uids >= 1,
        "action": "bloquear_si_falla",
    },
    {
        "check": "edge_cases_excluded_rows",
        "severity": "info",
        "value": len(edge_cases),
        "limit": "informativo",
        "pass": True,
        "action": "informativo",
    },
]
audit_summary = pd.DataFrame(audit_rows)
audit_summary.to_csv(OUTPUT_DIR / "label_audit_summary_v6_1.csv", index=False)
phrase_counts_df.to_csv(OUTPUT_DIR / "phrase_counts_v6_1.csv", index=False)

print("Resumen de auditoría V6")
display(audit_summary)
print("\nFrases negadas/hipotéticas/resueltas")
display(phrase_counts_df)

suspect_cols = first_existing_cols(df, [
    "uid", "filename", "projection", "diagnostic_category", "label", "structured_only_normal",
    "structured_has_no_indexing", "structured_only_no_indexing", "report_has_hypothetical_positive",
    "report_has_resolved_positive", "evidence_has_benign_nodular_context",
    "evidence_has_suspicious_nodular_context", "label_tokens", "problems", "mesh", "major_mesh",
    "findings", "impression", "text", "report_text", "image_path"
])

suspect_frames = []
for frame, reason in [
    (bad_normal_positive, "normal_tokens_labeled_positive"),
    (positive_no_indexing, "positive_no_indexing"),
    (positive_hypothetical, "positive_hypothetical"),
    (positive_resolved, "positive_resolved"),
    (positive_benign_nodular_context, "positive_benign_nodular_context"),
]:
    if len(frame):
        tmp = frame.copy()
        tmp["audit_reason"] = reason
        suspect_frames.append(tmp)

if suspect_frames:
    suspect_cases = pd.concat(suspect_frames, ignore_index=True)
    keep = ["audit_reason"] + suspect_cols
    keep = [c for c in keep if c in suspect_cases.columns]
    suspect_cases[keep].to_csv(OUTPUT_DIR / "label_audit_suspect_cases_v6_1.csv", index=False)
    print(f"Casos sospechosos guardados para revisión: {len(suspect_cases)}")
    display(suspect_cases[keep].head(50))
else:
    suspect_cases = pd.DataFrame()
    pd.DataFrame(columns=["audit_reason"] + suspect_cols).to_csv(OUTPUT_DIR / "label_audit_suspect_cases_v6_1.csv", index=False)
    print("No se detectaron casos sospechosos en los checks de auditoría.")

# Edge cases excluidos por diseño, para revisión manual.
if len(edge_cases):
    edge_keep = suspect_cols
    edge_cases[edge_keep].to_csv(OUTPUT_DIR / "edge_cases_to_review_v6_1.csv", index=False)
    print(f"Casos límite excluidos guardados: {len(edge_cases)}")
    display(edge_cases[edge_keep].head(30))
else:
    pd.DataFrame(columns=suspect_cols).to_csv(OUTPUT_DIR / "edge_cases_to_review_v6_1.csv", index=False)
    print("No se generaron edge cases con las categorías nuevas.")

# V6.1: archivo específico para los nodulares benignos/calcificados excluidos del binario principal.
benign_keep = [c for c in suspect_cols if c in benign_nodular_review_candidates.columns]
if len(benign_nodular_review_candidates):
    benign_nodular_review_candidates[benign_keep].to_csv(OUTPUT_DIR / "benign_nodular_review_candidates_v6_1.csv", index=False)
    print(f"Nodulares benignos/calcificados enviados a revisión: {len(benign_nodular_review_candidates)}")
else:
    pd.DataFrame(columns=benign_keep).to_csv(OUTPUT_DIR / "benign_nodular_review_candidates_v6_1.csv", index=False)
    print("No se detectaron nodulares benignos/calcificados para revisión específica.")

# Muestra para revisión manual: balanceada por categoría, incluyendo excluidos.
def sample_group(frame, condition, n, seed, reason):
    sub = frame[condition].copy()
    if len(sub) == 0:
        return sub
    sub = sub.drop_duplicates(subset=["uid"]).copy()
    out = sub.sample(n=min(n, len(sub)), random_state=seed).copy()
    out["manual_review_group"] = reason
    return out

review_frames = []
review_frames.append(sample_group(used_df, used_df["diagnostic_category"].eq("normal"), MANUAL_REVIEW_PER_GROUP, SEED + 1, "normal"))
review_frames.append(sample_group(used_df, used_df["diagnostic_category"].eq("acute_relevant"), MANUAL_REVIEW_PER_GROUP, SEED + 2, "acute_relevant"))
review_frames.append(sample_group(used_df, used_df["diagnostic_category"].eq("chronic_structural"), MANUAL_REVIEW_PER_GROUP, SEED + 3, "chronic_structural"))
review_frames.append(sample_group(used_df, used_df["diagnostic_category"].eq("nodular"), MANUAL_REVIEW_PER_GROUP, SEED + 4, "nodular"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("mild_ambiguous"), MANUAL_REVIEW_PER_GROUP, SEED + 5, "mild_ambiguous_excluded"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("incidental_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 6, "incidental_excluded"))
if "evidence_has_benign_nodular_context" in df.columns:
    review_frames.append(sample_group(df, df["evidence_has_benign_nodular_context"].eq(True), MANUAL_REVIEW_PER_GROUP, SEED + 61, "benign_nodular_context_review"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("discordant_review_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 7, "discordant_review_excluded"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("no_indexing_report_review_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 8, "no_indexing_report_review_excluded"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("no_indexing_unclear_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 9, "no_indexing_unclear_excluded"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("hypothetical_unclear_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 10, "hypothetical_unclear_excluded"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("resolved_nonactive_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 11, "resolved_nonactive_excluded"))
review_frames.append(sample_group(df, df["diagnostic_category"].eq("unclear_excluded"), MANUAL_REVIEW_PER_GROUP, SEED + 12, "unclear_excluded"))

manual_review = pd.concat([x for x in review_frames if len(x)], ignore_index=True) if any(len(x) for x in review_frames) else pd.DataFrame()
manual_review_cols = first_existing_cols(manual_review, [
    "manual_review_group", "uid", "filename", "projection", "diagnostic_category", "label",
    "structured_only_normal", "structured_has_no_indexing", "structured_only_no_indexing",
    "report_has_hypothetical_positive", "report_has_resolved_positive", "label_tokens",
    "problems", "mesh", "major_mesh", "findings", "impression", "text", "report_text", "image_path"
])
manual_review[manual_review_cols].to_csv(OUTPUT_DIR / "manual_review_sample_v6_1.csv", index=False)
print(f"Muestra de revisión manual guardada: {len(manual_review)} filas")
display(manual_review[manual_review_cols].head(40))

# Gate de seguridad.
audit_pass_all = bool(audit_summary["pass"].all())
critical_mask = audit_summary["severity"].eq("critical")
audit_gate_pass = bool(audit_summary.loc[critical_mask, "pass"].all())
review_failed = audit_summary[(audit_summary["severity"].eq("review")) & (~audit_summary["pass"])]
critical_failed = audit_summary[(audit_summary["severity"].eq("critical")) & (~audit_summary["pass"])]

print("\nAUDIT_PASS_ALL =", audit_pass_all)
print("AUDIT_GATE_PASS =", audit_gate_pass)
print("RUN_TRAINING_PHASE =", RUN_TRAINING_PHASE)
print("MANUAL_AUDIT_APPROVED =", MANUAL_AUDIT_APPROVED)

if len(review_failed):
    print("\nAviso: hay checks de revisión clínica que no pasaron. No bloquean entrenamiento automáticamente, pero revisa label_audit_suspect_cases_v6_1.csv.")
    display(review_failed)

if STRICT_LABEL_AUDIT and not audit_gate_pass:
    print("\nChecks críticos fallidos:")
    display(critical_failed)
    raise ValueError(
        "La auditoría crítica de etiquetas falló. Revisa label_audit_summary_v6_1.csv y "
        "label_audit_suspect_cases_v6_1.csv antes de entrenar."
    )

if RUN_TRAINING_PHASE and REQUIRE_MANUAL_AUDIT_APPROVAL_FOR_TRAINING and not MANUAL_AUDIT_APPROVED:
    raise ValueError(
        "El entrenamiento está activado, pero MANUAL_AUDIT_APPROVED=False. "
        "Revisa manual_review_sample_v6_1.csv y luego cambia MANUAL_AUDIT_APPROVED=True."
    )

if not RUN_TRAINING_PHASE:
    print("\nEntrenamiento desactivado por diseño. Revisa los CSV de auditoría; luego cambia RUN_TRAINING_PHASE=True si todo está bien.")
elif audit_gate_pass:
    print("\nLa compuerta crítica de auditoría pasó. El entrenamiento puede continuar.")


## 4. Particiones separadas por UID

V6.1 evita usar la misma validación para todo. El flujo principal queda así:

- `train_fit`: ajusta imagen y texto.
- `model_val`: early stopping del modelo visual.
- `calibration`: calibración de probabilidades y elección del peso de fusión.
- `threshold`: selección de umbrales operativos.
- `test`: evaluación final.

In [ ]:
if RUN_TRAINING_PHASE:
    def safe_stratify(labels):
        labels = pd.Series(labels)
        vc = labels.value_counts()
        if len(vc) < 2 or vc.min() < 2:
            return None
        return labels


    def uid_level_table(dataframe):
        return dataframe.groupby("uid").agg(
            label=("label", "max"),
            n_images=("label", "size"),
            categories=("diagnostic_category", lambda x: " | ".join(sorted(set(map(str, x))))),
        ).reset_index()


    def split_uid_table(uid_table, seed=42):
        fractions = {
            "test": TEST_FRACTION,
            "threshold": THRESHOLD_FRACTION,
            "calibration": CALIBRATION_FRACTION,
            "model_val": MODEL_VAL_FRACTION,
        }
        total = TRAIN_FIT_FRACTION + MODEL_VAL_FRACTION + CALIBRATION_FRACTION + THRESHOLD_FRACTION + TEST_FRACTION
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Las fracciones deben sumar 1.0. Suman {total}")

        remaining = uid_table.copy()
        remaining_fraction = 1.0
        parts = {}

        for i, (name, frac) in enumerate(fractions.items()):
            if frac <= 0:
                parts[name] = remaining.iloc[0:0].copy().reset_index(drop=True)
                continue
            relative_size = frac / remaining_fraction
            if relative_size <= 0 or relative_size >= 1:
                raise ValueError(f"Fracción inválida para {name}: frac={frac}, remaining_fraction={remaining_fraction}")
            strat = safe_stratify(remaining["label"])
            keep, split = train_test_split(
                remaining,
                test_size=relative_size,
                random_state=seed + i,
                stratify=strat,
            )
            parts[name] = split.reset_index(drop=True)
            remaining = keep.reset_index(drop=True)
            remaining_fraction -= frac

        parts["train_fit"] = remaining.reset_index(drop=True)
        return parts


    def materialize_splits(dataframe, uid_parts):
        split_dfs = {}
        for split_name, uid_df in uid_parts.items():
            uid_set = set(uid_df["uid"])
            split_dfs[split_name] = dataframe[dataframe["uid"].isin(uid_set)].reset_index(drop=True)
        return split_dfs

    uid_table = uid_level_table(used_df)
    uid_parts = split_uid_table(uid_table, seed=SEED)
    splits = materialize_splits(used_df, uid_parts)

    split_summary_rows = []
    for name in ["train_fit", "model_val", "calibration", "threshold", "test"]:
        sdf = splits[name]
        split_summary_rows.append({
            "split": name,
            "rows": len(sdf),
            "uids": sdf["uid"].nunique(),
            "row_prevalence": float(sdf["label"].mean()),
            "uid_prevalence": float(sdf.groupby("uid")["label"].max().mean()),
            "positives_rows": int((sdf["label"] == 1).sum()),
            "negatives_rows": int((sdf["label"] == 0).sum()),
        })

    split_summary_df = pd.DataFrame(split_summary_rows)
    split_summary_df.to_csv(OUTPUT_DIR / "split_summary_v6_1.csv", index=False)
    display(split_summary_df)

    # Sanity check: ningún UID debe aparecer en más de un split.
    seen = {}
    for name, sdf in splits.items():
        for uid in sdf["uid"].unique():
            if uid in seen:
                raise RuntimeError(f"Fuga de UID: {uid} aparece en {seen[uid]} y {name}")
            seen[uid] = name
    print("Split por UID validado: sin solapamiento.")

## 5. Dataset de imagen, transformaciones y modelo visual

Si `BACKBONE_NAME="xrv_densenet121"`, el modelo espera una imagen de 1 canal y usa normalización estilo `torchxrayvision`. Si se usa `torchvision`, trabaja con RGB e ImageNet mean/std.

In [ ]:
if RUN_TRAINING_PHASE:
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]


    def xrv_normalize_tensor(tensor):
        # tensor: [1, H, W], rango 0..1
        if HAS_XRV:
            arr = tensor.squeeze(0).numpy() * 255.0
            arr = xrv.datasets.normalize(arr, 255)
            return torch.from_numpy(arr).float().unsqueeze(0)
        # fallback razonable si xrv no está instalado
        return (tensor - 0.5) / 0.5


    def build_transforms(train=False):
        is_xrv = BACKBONE_NAME == "xrv_densenet121"
        pil_ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
        if train:
            pil_ops += [
                transforms.RandomRotation(degrees=4),
                transforms.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.97, 1.03)),
            ]

        if is_xrv:
            return transforms.Compose(pil_ops + [
                transforms.Grayscale(num_output_channels=1),
                transforms.ToTensor(),
                transforms.Lambda(xrv_normalize_tensor),
            ])

        return transforms.Compose(pil_ops + [
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])


    class IUXrayImageDataset(Dataset):
        def __init__(self, dataframe, transform=None):
            self.df = dataframe.reset_index(drop=True)
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            image = Image.open(row["image_path"]).convert("RGB")
            if self.transform:
                image = self.transform(image)
            label = torch.tensor(float(row["label"]), dtype=torch.float32)
            return {
                "image": image,
                "label": label,
                "idx": torch.tensor(idx, dtype=torch.long),
            }


    def make_loader(dataframe, train=False, batch_size=None):
        if batch_size is None:
            batch_size = BATCH_SIZE
        ds = IUXrayImageDataset(dataframe, transform=build_transforms(train=train))
        return DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=train,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )


    class ImageEncoder(nn.Module):
        def __init__(self, backbone_name="densenet121", pretrained=True, out_dim=256, unfreeze_last_block=True):
            super().__init__()
            self.backbone_name = backbone_name

            if backbone_name == "xrv_densenet121":
                if not HAS_XRV:
                    raise RuntimeError("torchxrayvision no está disponible.")
                weights = CXR_PRETRAINED_WEIGHTS if pretrained else None
                cnn = xrv.models.DenseNet(weights=weights)
                self.features = cnn.features
                in_dim = cnn.classifier.in_features
                self.target_layer = getattr(self.features, "denseblock4", self.features[-1])

                for p in self.features.parameters():
                    p.requires_grad = False
                if unfreeze_last_block:
                    for name, p in self.features.named_parameters():
                        if any(key in name for key in ["denseblock4", "norm5"]):
                            p.requires_grad = True

            elif backbone_name == "densenet121":
                if HAS_TORCHVISION_WEIGHTS:
                    weights = DenseNet121_Weights.DEFAULT if pretrained else None
                    cnn = densenet121(weights=weights)
                else:
                    cnn = densenet121(pretrained=pretrained)
                self.features = cnn.features
                in_dim = cnn.classifier.in_features
                self.target_layer = self.features.denseblock4

                for p in self.features.parameters():
                    p.requires_grad = False
                if unfreeze_last_block:
                    for p in self.features.denseblock4.parameters():
                        p.requires_grad = True
                    for p in self.features.norm5.parameters():
                        p.requires_grad = True

            elif backbone_name in {"resnet18", "resnet50"}:
                if backbone_name == "resnet18":
                    if HAS_TORCHVISION_WEIGHTS:
                        weights = ResNet18_Weights.DEFAULT if pretrained else None
                        cnn = resnet18(weights=weights)
                    else:
                        cnn = resnet18(pretrained=pretrained)
                else:
                    if HAS_TORCHVISION_WEIGHTS:
                        weights = ResNet50_Weights.DEFAULT if pretrained else None
                        cnn = resnet50(weights=weights)
                    else:
                        cnn = resnet50(pretrained=pretrained)
                modules = list(cnn.children())[:-2]
                self.features = nn.Sequential(*modules)
                in_dim = cnn.fc.in_features
                self.target_layer = cnn.layer4

                for p in self.features.parameters():
                    p.requires_grad = False
                if unfreeze_last_block:
                    # En Sequential, layer4 suele quedar como último bloque.
                    for p in self.features[-1].parameters():
                        p.requires_grad = True
            else:
                raise ValueError(f"Backbone no reconocido: {backbone_name}")

            self.proj = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(DROPOUT),
            )

        def forward(self, x):
            feats = self.features(x)
            feats = F.relu(feats, inplace=True)
            pooled = F.adaptive_avg_pool2d(feats, (1, 1)).flatten(1)
            return self.proj(pooled)


    class ImageOnlyClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.image_encoder = ImageEncoder(
                backbone_name=BACKBONE_NAME,
                pretrained=PRETRAINED,
                out_dim=FEATURE_DIM,
                unfreeze_last_block=UNFREEZE_LAST_BLOCK,
            )
            self.head = nn.Sequential(
                nn.LayerNorm(FEATURE_DIM),
                nn.Dropout(DROPOUT),
                nn.Linear(FEATURE_DIM, 1),
            )

        def forward(self, image):
            emb = self.image_encoder(image)
            return self.head(emb).squeeze(1)

## 6. Métricas, umbrales, calibración y utilidades estadísticas

Estas funciones se reutilizan en el experimento principal, en la validación cruzada y en la curva de aprendizaje.

In [ ]:
if RUN_TRAINING_PHASE:
    def safe_auc(y_true, probs):
        y_true = np.asarray(y_true).astype(int)
        probs = np.asarray(probs).astype(float)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, probs))


    def safe_pr_auc(y_true, probs):
        y_true = np.asarray(y_true).astype(int)
        probs = np.asarray(probs).astype(float)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(average_precision_score(y_true, probs))


    def safe_brier(y_true, probs):
        y_true = np.asarray(y_true).astype(int)
        probs = np.clip(np.asarray(probs).astype(float), 1e-6, 1 - 1e-6)
        if len(y_true) == 0:
            return np.nan
        return float(brier_score_loss(y_true, probs))


    def safe_logloss(y_true, probs):
        y_true = np.asarray(y_true).astype(int)
        probs = np.clip(np.asarray(probs).astype(float), 1e-6, 1 - 1e-6)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(log_loss(y_true, probs, labels=[0, 1]))


    def metrics_at_threshold(y_true, probs, threshold):
        y_true = np.asarray(y_true).astype(int)
        probs = np.asarray(probs).astype(float)
        preds = (probs >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
        specificity = tn / (tn + fp + 1e-8)
        fpr = fp / (fp + tn + 1e-8)
        fnr = fn / (fn + tp + 1e-8)
        return {
            "threshold": float(threshold),
            "accuracy": float(accuracy_score(y_true, preds)),
            "precision": float(precision_score(y_true, preds, zero_division=0)),
            "recall_sensitivity": float(recall_score(y_true, preds, zero_division=0)),
            "specificity": float(specificity),
            "false_positive_rate": float(fpr),
            "false_negative_rate": float(fnr),
            "f1": float(f1_score(y_true, preds, zero_division=0)),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }


    def full_metrics(y_true, probs, threshold):
        out = metrics_at_threshold(y_true, probs, threshold)
        out.update({
            "roc_auc": safe_auc(y_true, probs),
            "pr_auc": safe_pr_auc(y_true, probs),
            "brier": safe_brier(y_true, probs),
            "log_loss": safe_logloss(y_true, probs),
            "n": int(len(y_true)),
            "prevalence": float(np.mean(y_true)) if len(y_true) else np.nan,
        })
        return out


    def select_threshold_min_fpr_for_recall(y_true, probs, target_recall, grid_size=1001):
        y_true = np.asarray(y_true).astype(int)
        probs = np.asarray(probs).astype(float)
        grid = np.unique(np.concatenate([
            np.linspace(0.0, 1.0, grid_size),
            probs,
        ]))

        candidates = []
        for th in grid:
            m = metrics_at_threshold(y_true, probs, th)
            m["target_recall"] = float(target_recall)
            m["recall_constraint_met"] = bool(m["recall_sensitivity"] >= target_recall)
            candidates.append(m)

        ok = [m for m in candidates if m["recall_constraint_met"]]
        if ok:
            # Menor FPR; desempate: mayor threshold para no sobrepredecir, luego mayor F1.
            best = sorted(ok, key=lambda m: (m["false_positive_rate"], -m["threshold"], -m["f1"]))[0]
        else:
            # Si no se alcanza el recall objetivo, reportamos el mejor recall disponible y menor FPR entre empates.
            best = sorted(candidates, key=lambda m: (-m["recall_sensitivity"], m["false_positive_rate"], -m["threshold"]))[0]
        return best


    def logit_transform(p):
        p = np.clip(np.asarray(p).astype(float), 1e-6, 1 - 1e-6)
        return np.log(p / (1 - p))


    def fit_probability_calibrator(y_cal, p_cal, method="platt"):
        y_cal = np.asarray(y_cal).astype(int)
        p_cal = np.asarray(p_cal).astype(float)
        if len(np.unique(y_cal)) < 2:
            return {"method": "identity", "model": None}

        if method == "platt":
            model = LogisticRegression(max_iter=1000, solver="lbfgs")
            model.fit(logit_transform(p_cal).reshape(-1, 1), y_cal)
            return {"method": "platt", "model": model}

        if method == "isotonic":
            model = IsotonicRegression(out_of_bounds="clip")
            model.fit(p_cal, y_cal)
            return {"method": "isotonic", "model": model}

        raise ValueError("method debe ser 'platt' o 'isotonic'")


    def apply_probability_calibrator(calibrator, p):
        p = np.asarray(p).astype(float)
        method = calibrator.get("method", "identity")
        model = calibrator.get("model")
        if method == "identity" or model is None:
            return np.clip(p, 1e-6, 1 - 1e-6)
        if method == "platt":
            return model.predict_proba(logit_transform(p).reshape(-1, 1))[:, 1]
        if method == "isotonic":
            return np.clip(model.predict(p), 1e-6, 1 - 1e-6)
        raise ValueError(f"Calibrador no reconocido: {method}")


    def optimize_fusion_weight(y_cal, p_image, p_text, target_recall=0.90):
        """Selecciona un peso de fusión conservador.

        V6 optimizaba principalmente FPR en calibration y permitió bastante aporte textual.
        En V6.1 el texto se usa solo como complemento pequeño: el grid empieza en 0.85 de peso visual
        y la selección penaliza explícitamente mala calibración mediante Brier.
        """
        rows = []
        for w in FUSION_WEIGHT_GRID:
            p = w * np.asarray(p_image) + (1 - w) * np.asarray(p_text)
            m = select_threshold_min_fpr_for_recall(y_cal, p, target_recall, grid_size=THRESHOLD_GRID_SIZE)
            brier = safe_brier(y_cal, p)
            recall_shortfall = max(0.0, float(target_recall) - float(m["recall_sensitivity"]))
            selection_score = (
                float(m["false_positive_rate"]) +
                FUSION_BRIER_LAMBDA * float(brier) +
                FUSION_RECALL_SHORTFALL_PENALTY * (recall_shortfall ** 2)
            )
            rows.append({
                "image_weight": float(w),
                "text_weight": float(1 - w),
                "calibration_threshold": m["threshold"],
                "calibration_recall": m["recall_sensitivity"],
                "calibration_fpr": m["false_positive_rate"],
                "calibration_specificity": m["specificity"],
                "calibration_precision": m["precision"],
                "calibration_f1": m["f1"],
                "roc_auc": safe_auc(y_cal, p),
                "pr_auc": safe_pr_auc(y_cal, p),
                "brier": brier,
                "recall_shortfall": recall_shortfall,
                "selection_score": selection_score,
                "selection_mode": FUSION_SELECTION_MODE,
                "recall_constraint_met": m["recall_constraint_met"],
            })
        weight_df = pd.DataFrame(rows)
        ok_df = weight_df[weight_df["recall_constraint_met"]].copy()
        choose_df = ok_df if len(ok_df) else weight_df
        # Prioridad V6.1: score clínico penalizado, luego FPR, Brier, PR-AUC y mayor peso visual.
        best = choose_df.sort_values(
            ["selection_score", "calibration_fpr", "brier", "pr_auc", "image_weight"],
            ascending=[True, True, True, False, False],
        ).iloc[0].to_dict()
        return best, weight_df


    def aggregate_by_uid(pred_df, prob_cols):
        agg = {
            "label": ("label", "max"),
            "n_images": ("label", "size"),
            "categories": ("diagnostic_category", lambda x: " | ".join(sorted(set(map(str, x))))),
            "label_tokens": ("label_tokens", lambda x: " | ".join(sorted(set(" | ".join(map(str, x)).split(" | "))))),
        }
        for col in prob_cols:
            agg[col] = (col, "max")
        return pred_df.groupby("uid").agg(**agg).reset_index()


    def evaluate_thresholds_on_split(pred_df, model_prob_cols, thresholds_df, split_name):
        rows = []
        for model_name, prob_col in model_prob_cols.items():
            y = pred_df["label"].values.astype(int)
            p = pred_df[prob_col].values.astype(float)
            for _, th_row in thresholds_df[thresholds_df["model"] == model_name].iterrows():
                m = full_metrics(y, p, float(th_row["threshold"]))
                m.update({
                    "split": split_name,
                    "model": model_name,
                    "prob_col": prob_col,
                    "threshold_type": th_row["threshold_type"],
                    "target_recall": th_row["target_recall"],
                    "threshold_source": th_row.get("threshold_source", "threshold_split"),
                    "threshold_constraint_met_on_source": bool(th_row.get("recall_constraint_met", False)),
                })
                rows.append(m)
        return pd.DataFrame(rows)


    def exact_mcnemar_from_predictions(y_true, pred_a, pred_b, subset="negative"):
        y_true = np.asarray(y_true).astype(int)
        pred_a = np.asarray(pred_a).astype(int)
        pred_b = np.asarray(pred_b).astype(int)

        if subset == "negative":
            mask = y_true == 0
            # b: A se equivoca como FP, B corrige. c: A correcto, B nuevo FP.
            b = int(((pred_a[mask] == 1) & (pred_b[mask] == 0)).sum())
            c = int(((pred_a[mask] == 0) & (pred_b[mask] == 1)).sum())
        elif subset == "positive":
            mask = y_true == 1
            # b: A se equivoca como FN, B corrige. c: A correcto, B nuevo FN.
            b = int(((pred_a[mask] == 0) & (pred_b[mask] == 1)).sum())
            c = int(((pred_a[mask] == 1) & (pred_b[mask] == 0)).sum())
        else:
            raise ValueError("subset debe ser 'negative' o 'positive'")

        n = b + c
        if n == 0 or binomtest is None:
            p_value = np.nan
        else:
            p_value = float(binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue)
        return {"b_corrected_by_b": b, "c_new_errors_by_b": c, "n_discordant": n, "p_value": p_value}

## 7. Entrenamiento visual y pipeline central V6

La función `run_core_experiment` ejecuta el experimento completo sobre un conjunto de particiones. Esto permite reutilizarla para el holdout principal, para folds y para curva de aprendizaje.

In [ ]:
if RUN_TRAINING_PHASE:
    def forward_image(model, batch):
        image = batch["image"].to(device, non_blocking=True)
        return model(image=image)


    @torch.no_grad()
    def predict_image_proba(model, loader):
        model.eval()
        probs = []
        labels = []
        indices = []
        for batch in loader:
            logits = forward_image(model, batch)
            prob = torch.sigmoid(logits).detach().cpu().numpy()
            probs.extend(prob.tolist())
            labels.extend(batch["label"].cpu().numpy().astype(int).tolist())
            indices.extend(batch["idx"].cpu().numpy().astype(int).tolist())
        return np.array(labels), np.array(probs), np.array(indices)


    @torch.no_grad()
    def evaluate_image_loss(model, loader, criterion):
        model.eval()
        losses = []
        for batch in loader:
            labels = batch["label"].to(device, non_blocking=True)
            logits = forward_image(model, batch)
            loss = criterion(logits, labels)
            losses.append(float(loss.detach().cpu().item()))
        return float(np.mean(losses)) if losses else np.nan


    def train_image_model(train_df, val_df, run_tag="main", epochs=None, save_best=True):
        if epochs is None:
            epochs = EPOCHS_IMAGE

        train_loader = make_loader(train_df, train=True)
        val_loader = make_loader(val_df, train=False)

        num_pos = int((train_df["label"] == 1).sum())
        num_neg = int((train_df["label"] == 0).sum())
        pos_weight_value = num_neg / max(num_pos, 1)
        pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

        model = ImageOnlyClassifier().to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
        scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

        best_state = None
        best_score = -np.inf
        best_epoch = -1
        patience_counter = 0
        history = []

        print(f"\nEntrenando modelo visual [{run_tag}] | train={len(train_df)} val={len(val_df)} pos_weight={pos_weight_value:.3f}")

        for epoch in range(1, epochs + 1):
            model.train()
            train_losses = []
            pbar = tqdm(train_loader, desc=f"{run_tag} epoch {epoch}/{epochs}", leave=False)
            for batch in pbar:
                labels = batch["label"].to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                    logits = forward_image(model, batch)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                scaler.step(optimizer)
                scaler.update()
                train_losses.append(float(loss.detach().cpu().item()))
                pbar.set_postfix(loss=np.mean(train_losses))

            val_loss = evaluate_image_loss(model, val_loader, criterion)
            y_val, p_val, _ = predict_image_proba(model, val_loader)
            val_auc = safe_auc(y_val, p_val)
            val_pr = safe_pr_auc(y_val, p_val)
            if MODEL_SELECTION == "val_loss":
                monitor = -val_loss
            else:
                vals = [x for x in [val_auc, val_pr] if not np.isnan(x)]
                monitor = float(np.mean(vals)) if vals else -val_loss
            scheduler.step(monitor if not np.isnan(monitor) else -val_loss)

            row = {
                "run_tag": run_tag,
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)) if train_losses else np.nan,
                "val_loss": val_loss,
                "val_roc_auc": val_auc,
                "val_pr_auc": val_pr,
                "monitor": monitor,
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
            history.append(row)
            print(
                f"[{run_tag}] epoch {epoch:02d} | train_loss={row['train_loss']:.4f} "
                f"val_loss={val_loss:.4f} val_auc={val_auc:.4f} val_pr={val_pr:.4f} monitor={monitor:.4f}"
            )

            if monitor > best_score + 1e-5:
                best_score = monitor
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
                if save_best:
                    torch.save(best_state, OUTPUT_DIR / f"{run_tag}_best_image_model_v6_1.pt")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"Early stopping en epoch {epoch}. Mejor epoch: {best_epoch}")
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        history_df = pd.DataFrame(history)
        return model, history_df


    def attach_image_predictions(split_df, y, p, idx, prob_col="prob_image_raw"):
        pred_df = split_df.reset_index(drop=True).copy()
        pred_table = pd.DataFrame({"idx": idx, "y_pred_order": y, prob_col: p}).sort_values("idx")
        pred_df[prob_col] = pred_table[prob_col].values
        # Verificación suave de alineación.
        if not np.array_equal(pred_df["label"].values.astype(int), pred_table["y_pred_order"].values.astype(int)):
            raise RuntimeError("Las etiquetas no están alineadas con las predicciones de imagen.")
        return pred_df


    def build_text_classifier():
        return Pipeline([
            ("tfidf", TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=TFIDF_NGRAM_RANGE,
                max_features=TFIDF_MAX_FEATURES,
                min_df=1,
            )),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="liblinear",
                random_state=SEED,
            )),
        ])


    def build_stack_features(df):
        # Incluimos probabilidades y logits para que la regresión pueda corregir calibración.
        cols = []
        for col in ["prob_image_cal", "prob_text_cal"]:
            p = np.asarray(df[col].values, dtype=float)
            cols.append(p)
            cols.append(logit_transform(p))
        return np.column_stack(cols)


    def run_core_experiment(split_dfs, run_tag="main", epochs=None, save_outputs=True):
        set_seed(SEED)
        local_output_dir = OUTPUT_DIR / run_tag
        if save_outputs:
            local_output_dir.mkdir(parents=True, exist_ok=True)

        # 1) Imagen.
        image_model, history_df = train_image_model(
            split_dfs["train_fit"],
            split_dfs["model_val"],
            run_tag=run_tag,
            epochs=epochs,
            save_best=save_outputs,
        )

        pred_dfs = {}
        for split_name, sdf in split_dfs.items():
            loader = make_loader(sdf, train=False)
            y, p, idx = predict_image_proba(image_model, loader)
            pred_dfs[split_name] = attach_image_predictions(sdf, y, p, idx, prob_col="prob_image_raw")

        # 2) Texto TF-IDF entrenado solo con train_fit.
        text_clf = build_text_classifier()
        text_clf.fit(split_dfs["train_fit"]["text"].astype(str), split_dfs["train_fit"]["label"].astype(int))
        for split_name, pdf in pred_dfs.items():
            pdf["prob_text_raw"] = text_clf.predict_proba(pdf["text"].astype(str))[:, 1]

        # 3) Calibración explícita en split calibration.
        cal_df = pred_dfs["calibration"]
        image_calibrator = fit_probability_calibrator(cal_df["label"].values, cal_df["prob_image_raw"].values, method=CALIBRATION_METHOD)
        text_calibrator = fit_probability_calibrator(cal_df["label"].values, cal_df["prob_text_raw"].values, method=CALIBRATION_METHOD)

        for split_name, pdf in pred_dfs.items():
            pdf["prob_image_cal"] = apply_probability_calibrator(image_calibrator, pdf["prob_image_raw"].values)
            pdf["prob_text_cal"] = apply_probability_calibrator(text_calibrator, pdf["prob_text_raw"].values)

        # 4) Fusión ponderada conservadora. El peso se optimiza en calibration, no en threshold ni test.
        cal_df = pred_dfs["calibration"]
        best_weight, weight_search_df = optimize_fusion_weight(
            cal_df["label"].values,
            cal_df["prob_image_cal"].values,
            cal_df["prob_text_cal"].values,
            target_recall=PRIMARY_TARGET_RECALL,
        )
        w = float(best_weight["image_weight"])
        for split_name, pdf in pred_dfs.items():
            pdf["prob_fusion_weighted"] = w * pdf["prob_image_cal"].values + (1 - w) * pdf["prob_text_cal"].values

        # 5) Stacking calibrado como experimento secundario.
        stack_model = None
        if RUN_STACKING:
            stack_model = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=SEED)),
            ])
            X_cal = build_stack_features(pred_dfs["calibration"])
            y_cal = pred_dfs["calibration"]["label"].values.astype(int)
            if len(np.unique(y_cal)) >= 2:
                stack_model.fit(X_cal, y_cal)
                for split_name, pdf in pred_dfs.items():
                    pdf["prob_fusion_stack"] = stack_model.predict_proba(build_stack_features(pdf))[:, 1]
            else:
                print("Stacking omitido: calibration contiene una sola clase.")
                for split_name, pdf in pred_dfs.items():
                    pdf["prob_fusion_stack"] = pdf["prob_fusion_weighted"].values

        # 6) Modelos que se evaluarán.
        model_prob_cols = {
            "image_raw": "prob_image_raw",
            "image_calibrated": "prob_image_cal",
            "text_calibrated": "prob_text_cal",
            "fusion_weighted": "prob_fusion_weighted",
        }
        if RUN_STACKING:
            model_prob_cols["fusion_stack"] = "prob_fusion_stack"

        # 7) Selección de umbrales en split threshold.
        threshold_rows = []
        threshold_df_source = pred_dfs["threshold"]
        for model_name, prob_col in model_prob_cols.items():
            y_thr = threshold_df_source["label"].values.astype(int)
            p_thr = threshold_df_source[prob_col].values.astype(float)
            for target_recall in TARGET_RECALLS:
                selected = select_threshold_min_fpr_for_recall(
                    y_thr,
                    p_thr,
                    target_recall,
                    grid_size=THRESHOLD_GRID_SIZE,
                )
                selected.update({
                    "run_tag": run_tag,
                    "model": model_name,
                    "prob_col": prob_col,
                    "threshold_type": f"min_fpr_recall_{target_recall:.2f}",
                    "threshold_source": "threshold_split",
                    "roc_auc_source": safe_auc(y_thr, p_thr),
                    "pr_auc_source": safe_pr_auc(y_thr, p_thr),
                    "brier_source": safe_brier(y_thr, p_thr),
                    "log_loss_source": safe_logloss(y_thr, p_thr),
                })
                threshold_rows.append(selected)
        thresholds_df = pd.DataFrame(threshold_rows)

        # 8) Evaluación en test por imagen y por UID.
        test_pred_df = pred_dfs["test"].copy()
        summary_image = evaluate_thresholds_on_split(test_pred_df, model_prob_cols, thresholds_df, split_name="test_image")

        uid_test_pred_df = aggregate_by_uid(test_pred_df, list(model_prob_cols.values()))
        summary_uid = evaluate_thresholds_on_split(uid_test_pred_df, model_prob_cols, thresholds_df, split_name="test_uid")
        summary_df = pd.concat([summary_image, summary_uid], ignore_index=True)
        summary_df.insert(0, "run_tag", run_tag)

        # 9) Guardado.
        if save_outputs:
            history_df.to_csv(local_output_dir / f"training_history_{run_tag}_v6_1.csv", index=False)
            thresholds_df.to_csv(local_output_dir / f"thresholds_{run_tag}_v6_1.csv", index=False)
            summary_df.to_csv(local_output_dir / f"summary_{run_tag}_v6_1.csv", index=False)
            weight_search_df.to_csv(local_output_dir / f"fusion_weight_search_{run_tag}_v6_1.csv", index=False)
            with open(local_output_dir / f"fusion_weight_selected_{run_tag}_v6_1.json", "w") as f:
                json.dump(best_weight, f, indent=2)
            for split_name, pdf in pred_dfs.items():
                pdf.to_csv(local_output_dir / f"predictions_{split_name}_{run_tag}_v6_1.csv", index=False)
            uid_test_pred_df.to_csv(local_output_dir / f"predictions_test_uid_{run_tag}_v6_1.csv", index=False)

        return {
            "run_tag": run_tag,
            "image_model": image_model,
            "text_clf": text_clf,
            "stack_model": stack_model,
            "history_df": history_df,
            "pred_dfs": pred_dfs,
            "test_uid_pred_df": uid_test_pred_df,
            "thresholds_df": thresholds_df,
            "summary_df": summary_df,
            "model_prob_cols": model_prob_cols,
            "fusion_weight": best_weight,
            "fusion_weight_search_df": weight_search_df,
            "image_calibrator": image_calibrator,
            "text_calibrator": text_calibrator,
        }

## 8. Ejecución del experimento principal V6

Este es el resultado principal del notebook. Usa las particiones separadas y deja el test final sin tocar hasta la evaluación.

In [ ]:
if RUN_TRAINING_PHASE:
    main_result = run_core_experiment(splits, run_tag="main", epochs=EPOCHS_IMAGE, save_outputs=True)

    summary_df = main_result["summary_df"]
    thresholds_df = main_result["thresholds_df"]
    model_prob_cols = main_result["model_prob_cols"]
    test_pred_df = main_result["pred_dfs"]["test"]
    uid_test_pred_df = main_result["test_uid_pred_df"]
    history_df = main_result["history_df"]

    print("Peso de fusión seleccionado:")
    print(json.dumps(main_result["fusion_weight"], indent=2))

    primary_type = f"min_fpr_recall_{PRIMARY_TARGET_RECALL:.2f}"
    print("\nResumen test por imagen - umbral principal:")
    primary_summary_image = summary_df[(summary_df["split"] == "test_image") & (summary_df["threshold_type"] == primary_type)].copy()
    display(primary_summary_image[[
        "model", "roc_auc", "pr_auc", "brier", "log_loss", "threshold", "recall_sensitivity",
        "specificity", "false_positive_rate", "precision", "f1", "fp", "fn", "tp", "tn"
    ]].sort_values("model"))

    print("\nResumen test por UID - umbral principal:")
    primary_summary_uid = summary_df[(summary_df["split"] == "test_uid") & (summary_df["threshold_type"] == primary_type)].copy()
    display(primary_summary_uid[[
        "model", "roc_auc", "pr_auc", "brier", "log_loss", "threshold", "recall_sensitivity",
        "specificity", "false_positive_rate", "precision", "f1", "fp", "fn", "tp", "tn"
    ]].sort_values("model"))

## 9. Gráficas principales

Curvas ROC/PR, curvas de calibración y curvas por objetivo de sensibilidad.

In [ ]:
if RUN_TRAINING_PHASE:
    def plot_training_history(history_df, out_path):
        if history_df is None or len(history_df) == 0:
            return
        plt.figure(figsize=(7, 5))
        plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
        plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Curva de pérdida - modelo visual")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(out_path, dpi=180)
        plt.show()

        plt.figure(figsize=(7, 5))
        plt.plot(history_df["epoch"], history_df["val_roc_auc"], marker="o", label="val ROC-AUC")
        plt.plot(history_df["epoch"], history_df["val_pr_auc"], marker="o", label="val PR-AUC")
        plt.xlabel("Epoch")
        plt.ylabel("Score")
        plt.title("Curvas de validación - modelo visual")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(str(out_path).replace("loss", "auc_pr"), dpi=180)
        plt.show()


    def plot_roc_pr(pred_df, model_prob_cols, out_prefix):
        y = pred_df["label"].values.astype(int)

        plt.figure(figsize=(7, 6))
        for model_name, prob_col in model_prob_cols.items():
            p = pred_df[prob_col].values.astype(float)
            if len(np.unique(y)) < 2:
                continue
            fpr, tpr, _ = roc_curve(y, p)
            auc_score = safe_auc(y, p)
            plt.plot(fpr, tpr, label=f"{model_name} AUC={auc_score:.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--", label="azar")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate / Sensibilidad")
        plt.title("Curvas ROC - test")
        plt.legend(fontsize=8)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(f"{out_prefix}_roc_curves_test_v6_1.png", dpi=180)
        plt.show()

        plt.figure(figsize=(7, 6))
        for model_name, prob_col in model_prob_cols.items():
            p = pred_df[prob_col].values.astype(float)
            precision, recall, _ = precision_recall_curve(y, p)
            ap = safe_pr_auc(y, p)
            plt.plot(recall, precision, label=f"{model_name} AP={ap:.3f}")
        plt.xlabel("Recall / Sensibilidad")
        plt.ylabel("Precisión")
        plt.title("Curvas Precision-Recall - test")
        plt.legend(fontsize=8)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(f"{out_prefix}_precision_recall_curves_test_v6_1.png", dpi=180)
        plt.show()


    def plot_calibration(pred_df, model_prob_cols, out_path, n_bins=8):
        y = pred_df["label"].values.astype(int)
        plt.figure(figsize=(7, 6))
        for model_name, prob_col in model_prob_cols.items():
            p = np.clip(pred_df[prob_col].values.astype(float), 1e-6, 1 - 1e-6)
            if len(np.unique(y)) < 2:
                continue
            frac_pos, mean_pred = calibration_curve(y, p, n_bins=n_bins, strategy="quantile")
            plt.plot(mean_pred, frac_pos, marker="o", label=f"{model_name}")
        plt.plot([0, 1], [0, 1], linestyle="--", label="calibración perfecta")
        plt.xlabel("Probabilidad media predicha")
        plt.ylabel("Fracción positiva observada")
        plt.title("Curvas de calibración - test")
        plt.legend(fontsize=8)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(out_path, dpi=180)
        plt.show()


    def plot_metric_by_target(summary_df, metric, out_path, split="test_image"):
        sub = summary_df[summary_df["split"] == split].copy()
        sub = sub[sub["threshold_type"].str.startswith("min_fpr_recall_")]
        plt.figure(figsize=(7, 5))
        for model_name in sorted(sub["model"].unique()):
            mdf = sub[sub["model"] == model_name].sort_values("target_recall")
            plt.plot(mdf["target_recall"], mdf[metric], marker="o", label=model_name)
        plt.xlabel("Recall objetivo usado para seleccionar umbral")
        plt.ylabel(metric)
        plt.title(f"{metric} por recall objetivo - {split}")
        plt.legend(fontsize=8)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(out_path, dpi=180)
        plt.show()

    plot_training_history(history_df, OUTPUT_DIR / "main" / "image_loss_curve_v6_1.png")
    plot_roc_pr(test_pred_df, model_prob_cols, str(OUTPUT_DIR / "main"))
    plot_calibration(test_pred_df, model_prob_cols, OUTPUT_DIR / "main" / "calibration_curves_test_v6_1.png")
    plot_metric_by_target(summary_df, "false_positive_rate", OUTPUT_DIR / "main" / "fpr_by_target_recall_v6_1.png", split="test_image")
    plot_metric_by_target(summary_df, "recall_sensitivity", OUTPUT_DIR / "main" / "recall_by_target_recall_v6_1.png", split="test_image")
    plot_metric_by_target(summary_df, "specificity", OUTPUT_DIR / "main" / "specificity_by_target_recall_v6_1.png", split="test_image")

## 10. Bootstrap agrupado por UID y McNemar

Este bloque compara la fusión principal contra el modelo visual calibrado. El bootstrap se hace muestreando UIDs, no filas sueltas.

In [ ]:
if RUN_TRAINING_PHASE:
    def get_threshold_for(thresholds_df, model_name, threshold_type):
        rows = thresholds_df[(thresholds_df["model"] == model_name) & (thresholds_df["threshold_type"] == threshold_type)]
        if len(rows) == 0:
            raise ValueError(f"No hay threshold para {model_name} / {threshold_type}")
        return float(rows.iloc[0]["threshold"])


    def bootstrap_grouped_metric_delta(pred_df, model_a, model_b, model_prob_cols, thresholds_df, threshold_type, n_boot=1000, seed=123):
        rng = np.random.default_rng(seed)
        uids = np.array(sorted(pred_df["uid"].unique()))
        uid_to_idx = {uid: np.where(pred_df["uid"].values == uid)[0] for uid in uids}

        th_a = get_threshold_for(thresholds_df, model_a, threshold_type)
        th_b = get_threshold_for(thresholds_df, model_b, threshold_type)
        col_a = model_prob_cols[model_a]
        col_b = model_prob_cols[model_b]

        rows = []
        for _ in tqdm(range(n_boot), desc="Bootstrap UID", leave=False):
            sampled_uids = rng.choice(uids, size=len(uids), replace=True)
            idx = np.concatenate([uid_to_idx[uid] for uid in sampled_uids])
            sub = pred_df.iloc[idx]
            y = sub["label"].values.astype(int)
            ma = full_metrics(y, sub[col_a].values, th_a)
            mb = full_metrics(y, sub[col_b].values, th_b)
            rows.append({
                "delta_fpr_b_minus_a": mb["false_positive_rate"] - ma["false_positive_rate"],
                "delta_recall_b_minus_a": mb["recall_sensitivity"] - ma["recall_sensitivity"],
                "delta_specificity_b_minus_a": mb["specificity"] - ma["specificity"],
                "delta_precision_b_minus_a": mb["precision"] - ma["precision"],
                "delta_brier_b_minus_a": mb["brier"] - ma["brier"],
                "delta_fp_b_minus_a": mb["fp"] - ma["fp"],
                "delta_fn_b_minus_a": mb["fn"] - ma["fn"],
            })
        boot_df = pd.DataFrame(rows)
        ci_rows = []
        for col in boot_df.columns:
            ci_rows.append({
                "metric": col,
                "mean": float(boot_df[col].mean()),
                "p2_5": float(np.percentile(boot_df[col], 2.5)),
                "p50": float(np.percentile(boot_df[col], 50)),
                "p97_5": float(np.percentile(boot_df[col], 97.5)),
            })
        return boot_df, pd.DataFrame(ci_rows)

    primary_type = f"min_fpr_recall_{PRIMARY_TARGET_RECALL:.2f}"
    comparison_rows = []
    bootstrap_outputs = []

    for fusion_model in [m for m in ["fusion_weighted", "fusion_stack"] if m in model_prob_cols]:
        boot_df, ci_df = bootstrap_grouped_metric_delta(
            test_pred_df,
            PRIMARY_IMAGE_MODEL,
            fusion_model,
            model_prob_cols,
            thresholds_df,
            primary_type,
            n_boot=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED,
        )
        boot_df.insert(0, "fusion_model", fusion_model)
        ci_df.insert(0, "fusion_model", fusion_model)
        bootstrap_outputs.append(boot_df)
        comparison_rows.append(ci_df)

    bootstrap_all_df = pd.concat(bootstrap_outputs, ignore_index=True) if bootstrap_outputs else pd.DataFrame()
    bootstrap_ci_df = pd.concat(comparison_rows, ignore_index=True) if comparison_rows else pd.DataFrame()
    bootstrap_all_df.to_csv(OUTPUT_DIR / "main" / "bootstrap_grouped_uid_deltas_v6_1.csv", index=False)
    bootstrap_ci_df.to_csv(OUTPUT_DIR / "main" / "bootstrap_grouped_uid_ci_v6_1.csv", index=False)

    print("IC bootstrap agrupado por UID. Diferencia = fusión - imagen_calibrated")
    display(bootstrap_ci_df)

    # McNemar exacto para FPR y FNR.
    mcnemar_rows = []
    for fusion_model in [m for m in ["fusion_weighted", "fusion_stack"] if m in model_prob_cols]:
        th_img = get_threshold_for(thresholds_df, PRIMARY_IMAGE_MODEL, primary_type)
        th_fus = get_threshold_for(thresholds_df, fusion_model, primary_type)
        pred_img = (test_pred_df[model_prob_cols[PRIMARY_IMAGE_MODEL]].values >= th_img).astype(int)
        pred_fus = (test_pred_df[model_prob_cols[fusion_model]].values >= th_fus).astype(int)
        y = test_pred_df["label"].values.astype(int)
        fpr_test = exact_mcnemar_from_predictions(y, pred_img, pred_fus, subset="negative")
        fnr_test = exact_mcnemar_from_predictions(y, pred_img, pred_fus, subset="positive")
        mcnemar_rows.append({"comparison": f"{PRIMARY_IMAGE_MODEL}_vs_{fusion_model}", "subset": "negative_FPR", **fpr_test})
        mcnemar_rows.append({"comparison": f"{PRIMARY_IMAGE_MODEL}_vs_{fusion_model}", "subset": "positive_FNR", **fnr_test})

    mcnemar_df = pd.DataFrame(mcnemar_rows)
    mcnemar_df.to_csv(OUTPUT_DIR / "main" / "mcnemar_primary_threshold_v6_1.csv", index=False)
    display(mcnemar_df)

## 11. Sensibilidad igualada en test — análisis oráculo

Este bloque **no** se usa como resultado desplegable, porque el umbral se elige mirando test. Sirve solo para diagnosticar si la curva de la fusión es realmente mejor que la imagen a sensibilidad comparable.

In [ ]:
if RUN_TRAINING_PHASE:
    oracle_rows = []
    for model_name, prob_col in model_prob_cols.items():
        y = test_pred_df["label"].values.astype(int)
        p = test_pred_df[prob_col].values.astype(float)
        for target_recall in TARGET_RECALLS:
            selected = select_threshold_min_fpr_for_recall(y, p, target_recall, grid_size=THRESHOLD_GRID_SIZE)
            selected.update({
                "model": model_name,
                "prob_col": prob_col,
                "target_recall": target_recall,
                "threshold_type": f"oracle_test_min_fpr_recall_{target_recall:.2f}",
                "roc_auc": safe_auc(y, p),
                "pr_auc": safe_pr_auc(y, p),
                "brier": safe_brier(y, p),
            })
            oracle_rows.append(selected)

    oracle_df = pd.DataFrame(oracle_rows)
    oracle_df.to_csv(OUTPUT_DIR / "main" / "diagnostic_test_oracle_equal_recall_v6_1.csv", index=False)
    display(oracle_df[["model", "target_recall", "threshold", "recall_sensitivity", "specificity", "false_positive_rate", "precision", "fp", "fn", "recall_constraint_met"]])

## 12. Análisis de errores y transiciones

Aquí buscamos qué casos corrige o empeora la fusión frente al modelo visual calibrado.

In [ ]:
if RUN_TRAINING_PHASE:
    def add_predictions_and_errors(pred_df, model_prob_cols, thresholds_df, threshold_type):
        out = pred_df.copy()
        y = out["label"].values.astype(int)
        for model_name, prob_col in model_prob_cols.items():
            th = get_threshold_for(thresholds_df, model_name, threshold_type)
            pred = (out[prob_col].values >= th).astype(int)
            out[f"pred_{model_name}"] = pred
            out[f"error_{model_name}"] = np.where(
                (y == 0) & (pred == 1), "FP",
                np.where((y == 1) & (pred == 0), "FN", "OK")
            )
        return out

    analysis_df = add_predictions_and_errors(test_pred_df, model_prob_cols, thresholds_df, primary_type)

    for fusion_model in [m for m in ["fusion_weighted", "fusion_stack"] if m in model_prob_cols]:
        analysis_df[f"transition_{PRIMARY_IMAGE_MODEL}_to_{fusion_model}"] = (
            analysis_df[f"error_{PRIMARY_IMAGE_MODEL}"] + "_to_" + analysis_df[f"error_{fusion_model}"]
        )

    analysis_df.to_csv(OUTPUT_DIR / "main" / "error_analysis_test_v6_1.csv", index=False)

    error_counts = []
    for model_name in model_prob_cols.keys():
        vc = analysis_df[f"error_{model_name}"].value_counts().to_dict()
        error_counts.append({"model": model_name, **vc})
    error_counts_df = pd.DataFrame(error_counts).fillna(0)
    error_counts_df.to_csv(OUTPUT_DIR / "main" / "error_counts_primary_threshold_v6_1.csv", index=False)
    print("Errores por modelo en el umbral principal:")
    display(error_counts_df)

    transition_rows = []
    for fusion_model in [m for m in ["fusion_weighted", "fusion_stack"] if m in model_prob_cols]:
        col = f"transition_{PRIMARY_IMAGE_MODEL}_to_{fusion_model}"
        vc = analysis_df[col].value_counts().reset_index()
        vc.columns = ["transition", "count"]
        vc.insert(0, "fusion_model", fusion_model)
        transition_rows.append(vc)
    transition_df = pd.concat(transition_rows, ignore_index=True) if transition_rows else pd.DataFrame()
    transition_df.to_csv(OUTPUT_DIR / "main" / "error_transitions_v6_1.csv", index=False)
    print("\nTransiciones imagen calibrada -> fusión:")
    display(transition_df)

    # Tokens deduplicados por caso.
    label_error_rows = []
    for model_name in model_prob_cols.keys():
        for err in ["FP", "FN"]:
            sub = analysis_df[analysis_df[f"error_{model_name}"] == err]
            token_counter = Counter()
            for txt in sub["label_tokens"].astype(str):
                tokens = [t.strip() for t in str(txt).split(" | ") if t.strip()]
                for tok in sorted(set(tokens)):
                    token_counter[tok] += 1
            for tok, count in token_counter.most_common(25):
                label_error_rows.append({"model": model_name, "error_type": err, "label_token": tok, "count": count})

    label_error_df = pd.DataFrame(label_error_rows)
    label_error_df.to_csv(OUTPUT_DIR / "main" / "top_label_tokens_by_error_v6_1.csv", index=False)
    print("\nTokens de etiqueta más frecuentes en errores:")
    display(label_error_df.head(80))

    # Casos erróneos de mayor confianza.
    hard_cases = []
    for model_name, prob_col in model_prob_cols.items():
        sub = analysis_df[analysis_df[f"error_{model_name}"].isin(["FP", "FN"])].copy()
        sub["error_confidence"] = np.where(sub[f"error_{model_name}"] == "FP", sub[prob_col], 1 - sub[prob_col])
        cols = [
            "uid", "filename", "diagnostic_category", "label", prob_col,
            f"pred_{model_name}", f"error_{model_name}", "error_confidence",
            "label_tokens", "text", "image_path",
        ]
        top = sub.sort_values("error_confidence", ascending=False)[cols].head(30).copy()
        top.insert(0, "model", model_name)
        hard_cases.append(top)

    hard_cases_df = pd.concat(hard_cases, ignore_index=True) if hard_cases else pd.DataFrame()
    hard_cases_df.to_csv(OUTPUT_DIR / "main" / "hardest_error_cases_v6_1.csv", index=False)
    print("\nCasos erróneos con mayor confianza:")
    display(hard_cases_df.head(60))

## 13. Interpretación automática prudente

Este texto se guarda como `.txt`. Úsalo como borrador, no como conclusión final sin revisión humana.

In [ ]:
if RUN_TRAINING_PHASE:
    def get_primary_row(summary_df, model_name, split="test_image"):
        rows = summary_df[(summary_df["split"] == split) & (summary_df["model"] == model_name) & (summary_df["threshold_type"] == primary_type)]
        if len(rows) == 0:
            return None
        return rows.iloc[0].to_dict()

    image_m = get_primary_row(summary_df, PRIMARY_IMAGE_MODEL, split="test_image")
    fusion_m = get_primary_row(summary_df, PRIMARY_FUSION_MODEL, split="test_image")

    lines = []
    lines.append("Interpretación automática V6")
    lines.append("============================")
    lines.append("")
    lines.append(f"Tarea clínica evaluada: {TASK_MODE}")
    lines.append(f"Backbone visual: {BACKBONE_NAME}")
    lines.append(f"Calibración: {CALIBRATION_METHOD}")
    lines.append(f"Umbral principal: {primary_type}, seleccionado en split threshold y aplicado en test.")
    lines.append("")

    if image_m is not None:
        lines.append(f"Modelo visual calibrado ({PRIMARY_IMAGE_MODEL}) en test por imagen:")
        lines.append(f"- ROC-AUC: {image_m['roc_auc']:.4f}")
        lines.append(f"- PR-AUC: {image_m['pr_auc']:.4f}")
        lines.append(f"- Brier: {image_m['brier']:.4f}")
        lines.append(f"- Sensibilidad: {image_m['recall_sensitivity']:.4f}")
        lines.append(f"- Especificidad: {image_m['specificity']:.4f}")
        lines.append(f"- FPR: {image_m['false_positive_rate']:.4f}")
        lines.append(f"- FP/FN: {int(image_m['fp'])}/{int(image_m['fn'])}")
        lines.append("")

    if fusion_m is not None and image_m is not None:
        fpr_diff = fusion_m["false_positive_rate"] - image_m["false_positive_rate"]
        recall_diff = fusion_m["recall_sensitivity"] - image_m["recall_sensitivity"]
        auc_diff = fusion_m["roc_auc"] - image_m["roc_auc"]
        pr_diff = fusion_m["pr_auc"] - image_m["pr_auc"]
        brier_diff = fusion_m["brier"] - image_m["brier"]

        lines.append(f"Fusión principal ({PRIMARY_FUSION_MODEL}) en test por imagen:")
        lines.append(f"- ROC-AUC: {fusion_m['roc_auc']:.4f} ({auc_diff:+.4f} vs imagen calibrada)")
        lines.append(f"- PR-AUC: {fusion_m['pr_auc']:.4f} ({pr_diff:+.4f} vs imagen calibrada)")
        lines.append(f"- Brier: {fusion_m['brier']:.4f} ({brier_diff:+.4f} vs imagen calibrada)")
        lines.append(f"- Sensibilidad: {fusion_m['recall_sensitivity']:.4f} ({recall_diff:+.4f} vs imagen calibrada)")
        lines.append(f"- Especificidad: {fusion_m['specificity']:.4f}")
        lines.append(f"- FPR: {fusion_m['false_positive_rate']:.4f} ({fpr_diff:+.4f} vs imagen calibrada)")
        lines.append(f"- FP/FN: {int(fusion_m['fp'])}/{int(fusion_m['fn'])}")
        lines.append("")

        lines.append("Conclusión prudente:")
        if fpr_diff < 0 and recall_diff >= -0.02:
            lines.append("La fusión reduce falsos positivos con pérdida de sensibilidad nula o muy pequeña. Esto apoya su uso como calibración/filtro operativo, pero debe revisarse si también mejora AUC/PR-AUC y si el resultado se sostiene en cross-validation.")
        elif fpr_diff < 0 and recall_diff < -0.02:
            lines.append("La fusión reduce falsos positivos, pero lo hace sacrificando sensibilidad. Debe presentarse como trade-off clínico, no como mejora global.")
        elif fpr_diff >= 0 and recall_diff > 0:
            lines.append("La fusión aumenta sensibilidad, pero no reduce falsos positivos. Puede ser útil para screening sensible, no como filtro de FP.")
        else:
            lines.append("La fusión no muestra una mejora clara frente a imagen calibrada en el punto operativo principal. Conviene revisar el aporte del texto, el target clínico y la estabilidad por folds.")

    lines.append("")
    lines.append("Notas metodológicas:")
    lines.append("- Las particiones se hacen por UID para evitar fuga entre imágenes del mismo caso.")
    lines.append("- La rama textual usa solo indication; findings/impression no entran como input del modelo.")
    lines.append("- Calibration y threshold son splits separados; esto reduce sobreajuste de umbral.")
    lines.append("- El análisis oráculo de sensibilidad igualada en test es diagnóstico, no resultado operativo.")
    lines.append("- Para una conclusión final robusta, activar RUN_KFOLD_CV=True y revisar estabilidad de la reducción de FPR.")

    interpretation = "\n".join(lines)
    print(interpretation)
    with open(OUTPUT_DIR / "main" / "automatic_interpretation_v6_1.txt", "w") as f:
        f.write(interpretation)

## 14. Bloque opcional: curva de aprendizaje

Activa `RUN_LEARNING_CURVE=True` para comprobar si el modelo sigue mejorando al aumentar los datos. Si la curva todavía sube al 100%, el proyecto está claramente limitado por tamaño de base.

In [ ]:
if RUN_TRAINING_PHASE:
    def sample_train_fraction_by_uid(train_df, fraction, seed):
        if fraction >= 0.999:
            return train_df.reset_index(drop=True)
        uid_df = uid_level_table(train_df)
        strat = safe_stratify(uid_df["label"])
        train_sample, _ = train_test_split(
            uid_df,
            train_size=fraction,
            random_state=seed,
            stratify=strat,
        )
        return train_df[train_df["uid"].isin(set(train_sample["uid"]))].reset_index(drop=True)

    learning_curve_rows = []

    if RUN_LEARNING_CURVE:
        print("Ejecutando curva de aprendizaje. Esto puede tardar bastante.")
        for frac in LEARNING_CURVE_FRACTIONS:
            for rep in range(LEARNING_CURVE_REPEATS):
                run_tag = f"lc_frac_{frac:.2f}_rep_{rep+1}"
                sampled_train = sample_train_fraction_by_uid(splits["train_fit"], frac, seed=SEED + rep + int(frac * 1000))
                lc_splits = dict(splits)
                lc_splits["train_fit"] = sampled_train
                print(f"\nCurva aprendizaje {run_tag}: train rows={len(sampled_train)} uids={sampled_train['uid'].nunique()}")
                lc_result = run_core_experiment(lc_splits, run_tag=run_tag, epochs=EPOCHS_LEARNING_CURVE, save_outputs=False)
                lc_summary = lc_result["summary_df"]
                take = lc_summary[(lc_summary["split"] == "test_image") & (lc_summary["threshold_type"] == primary_type)].copy()
                take["train_fraction"] = frac
                take["repeat"] = rep + 1
                take["train_rows"] = len(sampled_train)
                take["train_uids"] = sampled_train["uid"].nunique()
                learning_curve_rows.append(take)
                del lc_result
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        learning_curve_df = pd.concat(learning_curve_rows, ignore_index=True) if learning_curve_rows else pd.DataFrame()
        learning_curve_df.to_csv(OUTPUT_DIR / "learning_curve_results_v6_1.csv", index=False)
        display(learning_curve_df[["train_fraction", "repeat", "model", "roc_auc", "pr_auc", "recall_sensitivity", "false_positive_rate", "brier"]].head(30))

        # Gráfica simple de AUC vs fracción para modelos principales.
        plt.figure(figsize=(7, 5))
        for model_name in [PRIMARY_IMAGE_MODEL, PRIMARY_FUSION_MODEL]:
            sub = learning_curve_df[learning_curve_df["model"] == model_name]
            if len(sub) == 0:
                continue
            grouped = sub.groupby("train_fraction")["roc_auc"].agg(["mean", "std"]).reset_index()
            plt.errorbar(grouped["train_fraction"], grouped["mean"], yerr=grouped["std"], marker="o", capsize=3, label=model_name)
        plt.xlabel("Fracción de train_fit usada")
        plt.ylabel("ROC-AUC en test")
        plt.title("Curva de aprendizaje - ROC-AUC")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "learning_curve_auc_v6_1.png", dpi=180)
        plt.show()
    else:
        print("RUN_LEARNING_CURVE=False. Actívalo en configuración para ejecutar este bloque.")

## 15. Bloque opcional: 5-fold cross-validation por UID

Activa `RUN_KFOLD_CV=True` para estimar estabilidad del resultado. Cada fold entrena un modelo visual nuevo, así que este bloque es costoso.

In [ ]:
if RUN_TRAINING_PHASE:
    def make_inner_splits_for_outer_train(outer_train_df, seed):
        outer_uid_table = uid_level_table(outer_train_df)
        # Dentro del outer train se crean train_fit/model_val/calibration/threshold.
        # Reescalamos fracciones excluyendo test.
        original = (TRAIN_FIT_FRACTION, MODEL_VAL_FRACTION, CALIBRATION_FRACTION, THRESHOLD_FRACTION, TEST_FRACTION)
        try:
            globals()["TEST_FRACTION"] = 0.0
            total_dev = TRAIN_FIT_FRACTION + MODEL_VAL_FRACTION + CALIBRATION_FRACTION + THRESHOLD_FRACTION
            globals()["TRAIN_FIT_FRACTION"] = TRAIN_FIT_FRACTION / total_dev
            globals()["MODEL_VAL_FRACTION"] = MODEL_VAL_FRACTION / total_dev
            globals()["CALIBRATION_FRACTION"] = CALIBRATION_FRACTION / total_dev
            globals()["THRESHOLD_FRACTION"] = THRESHOLD_FRACTION / total_dev
            uid_parts_inner = split_uid_table(outer_uid_table, seed=seed)
        finally:
            globals()["TRAIN_FIT_FRACTION"], globals()["MODEL_VAL_FRACTION"], globals()["CALIBRATION_FRACTION"], globals()["THRESHOLD_FRACTION"], globals()["TEST_FRACTION"] = original

        inner = materialize_splits(outer_train_df, uid_parts_inner)
        # split_uid_table con TEST_FRACTION=0 puede generar test vacío; lo descartamos.
        inner = {k: v for k, v in inner.items() if k in {"train_fit", "model_val", "calibration", "threshold"}}
        return inner

    cv_rows = []

    if RUN_KFOLD_CV:
        print("Ejecutando StratifiedGroupKFold por UID. Esto puede tardar bastante.")
        uid_df_cv = uid_level_table(used_df)
        sgkf = StratifiedGroupKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=SEED)

        fold_iter = sgkf.split(uid_df_cv, uid_df_cv["label"].values, groups=uid_df_cv["uid"].values)
        for fold_idx, (train_idx, test_idx) in enumerate(fold_iter, start=1):
            if fold_idx > MAX_FOLDS_TO_RUN:
                break
            run_tag = f"cv_fold_{fold_idx}"
            outer_train_uids = set(uid_df_cv.iloc[train_idx]["uid"])
            outer_test_uids = set(uid_df_cv.iloc[test_idx]["uid"])
            outer_train_df = used_df[used_df["uid"].isin(outer_train_uids)].reset_index(drop=True)
            outer_test_df = used_df[used_df["uid"].isin(outer_test_uids)].reset_index(drop=True)

            inner_splits = make_inner_splits_for_outer_train(outer_train_df, seed=SEED + fold_idx)
            inner_splits["test"] = outer_test_df

            print(f"\nFold {fold_idx}: train={len(outer_train_df)} test={len(outer_test_df)}")
            fold_result = run_core_experiment(inner_splits, run_tag=run_tag, epochs=EPOCHS_CV, save_outputs=False)
            fold_summary = fold_result["summary_df"].copy()
            fold_summary["fold"] = fold_idx
            cv_rows.append(fold_summary)
            del fold_result
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        cv_summary_df = pd.concat(cv_rows, ignore_index=True) if cv_rows else pd.DataFrame()
        cv_summary_df.to_csv(OUTPUT_DIR / "cv_summary_v6_1.csv", index=False)

        cv_primary = cv_summary_df[(cv_summary_df["split"] == "test_image") & (cv_summary_df["threshold_type"] == primary_type)].copy()
        display(cv_primary[["fold", "model", "roc_auc", "pr_auc", "recall_sensitivity", "specificity", "false_positive_rate", "brier", "fp", "fn"]])

        cv_agg = cv_primary.groupby("model").agg(
            roc_auc_mean=("roc_auc", "mean"),
            roc_auc_std=("roc_auc", "std"),
            pr_auc_mean=("pr_auc", "mean"),
            pr_auc_std=("pr_auc", "std"),
            recall_mean=("recall_sensitivity", "mean"),
            recall_std=("recall_sensitivity", "std"),
            fpr_mean=("false_positive_rate", "mean"),
            fpr_std=("false_positive_rate", "std"),
            brier_mean=("brier", "mean"),
            brier_std=("brier", "std"),
        ).reset_index()
        cv_agg.to_csv(OUTPUT_DIR / "cv_primary_aggregate_v6_1.csv", index=False)
        display(cv_agg)

        plt.figure(figsize=(7, 5))
        for model_name in [PRIMARY_IMAGE_MODEL, PRIMARY_FUSION_MODEL]:
            sub = cv_primary[cv_primary["model"] == model_name].sort_values("fold")
            if len(sub) == 0:
                continue
            plt.plot(sub["fold"], sub["false_positive_rate"], marker="o", label=model_name)
        plt.xlabel("Fold")
        plt.ylabel("FPR")
        plt.title("FPR por fold - umbral principal")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "cv_fpr_by_fold_v6_1.png", dpi=180)
        plt.show()
    else:
        print("RUN_KFOLD_CV=False. Actívalo en configuración para ejecutar este bloque.")

## 16. Grad-CAM opcional

Úsalo solo para inspeccionar casos concretos. No debe ser la base de la conclusión cuantitativa.

In [ ]:
if RUN_TRAINING_PHASE:
    if RUN_GRADCAM:
        try:
            from pytorch_grad_cam import GradCAM
            from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
            from pytorch_grad_cam.utils.image import show_cam_on_image

            gradcam_dir = OUTPUT_DIR / "main" / "gradcam_v6_1"
            gradcam_dir.mkdir(parents=True, exist_ok=True)

            image_model = main_result["image_model"]
            image_model.eval()

            # Seleccionamos errores visuales calibrados de mayor confianza.
            th_image = get_threshold_for(thresholds_df, PRIMARY_IMAGE_MODEL, primary_type)
            cam_cases = analysis_df[analysis_df[f"error_{PRIMARY_IMAGE_MODEL}"].isin(["FP", "FN"])].copy()
            cam_cases["cam_priority"] = np.where(
                cam_cases[f"error_{PRIMARY_IMAGE_MODEL}"] == "FP",
                cam_cases[model_prob_cols[PRIMARY_IMAGE_MODEL]],
                1 - cam_cases[model_prob_cols[PRIMARY_IMAGE_MODEL]],
            )
            cam_cases = cam_cases.sort_values("cam_priority", ascending=False).head(8)

            class ImageModelWrapper(nn.Module):
                def __init__(self, model):
                    super().__init__()
                    self.model = model
                def forward(self, x):
                    return self.model(x).unsqueeze(1)

            wrapped = ImageModelWrapper(image_model).to(device).eval()
            target_layers = [image_model.image_encoder.target_layer]
            cam = GradCAM(model=wrapped, target_layers=target_layers)

            raw_transform = transforms.Compose([
                transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                transforms.ToTensor(),
            ])
            norm_transform = build_transforms(train=False)

            saved = []
            for _, row in cam_cases.iterrows():
                img_pil = Image.open(row["image_path"]).convert("RGB")
                rgb = raw_transform(img_pil).permute(1, 2, 0).numpy()
                input_tensor = norm_transform(img_pil).unsqueeze(0).to(device)
                target_value = 1 if row[f"error_{PRIMARY_IMAGE_MODEL}"] == "FP" else 0
                grayscale_cam = cam(input_tensor=input_tensor, targets=[BinaryClassifierOutputTarget(target_value)])[0]
                visualization = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)
                out_name = f"{row[f'error_{PRIMARY_IMAGE_MODEL}']}_uid_{row['uid']}_{row['filename']}.png".replace("/", "_")
                out_path = gradcam_dir / out_name
                Image.fromarray(visualization).save(out_path)
                saved.append(str(out_path))

            print("Grad-CAM guardados:")
            for p in saved:
                print(p)
        except Exception as e:
            print("No se pudo ejecutar Grad-CAM. El experimento principal no depende de esto.")
            print(type(e).__name__, e)
    else:
        print("RUN_GRADCAM=False. Actívalo si necesitas mapas de calor.")

## 17. Manifiesto reproducible V6

Esta celda guarda un manifiesto simple con configuración, tamaño del dataset y archivos principales. Sirve para revisar rápidamente si el ZIP final corresponde a la corrida esperada.


In [ ]:
manifest = {
    "version": VERSION,
    "task_mode": TASK_MODE,
    "seed": SEED,
    "run_training_phase": RUN_TRAINING_PHASE,
    "manual_audit_approved": MANUAL_AUDIT_APPROVED,
    "audit_pass_all": bool(audit_pass_all) if "audit_pass_all" in globals() else None,
    "audit_gate_pass": bool(audit_gate_pass) if "audit_gate_pass" in globals() else None,
    "run_kfold_cv": RUN_KFOLD_CV,
    "run_learning_curve": RUN_LEARNING_CURVE,
    "run_gradcam": RUN_GRADCAM,
    "dataset_rows_used": int(len(used_df)) if "used_df" in globals() else None,
    "dataset_uids_used": int(used_df["uid"].nunique()) if "used_df" in globals() else None,
    "row_prevalence": float(used_df["label"].mean()) if "used_df" in globals() and len(used_df) else None,
    "uid_prevalence": float(used_df.groupby("uid")["label"].max().mean()) if "used_df" in globals() and len(used_df) else None,
    "primary_image_model": PRIMARY_IMAGE_MODEL,
    "primary_fusion_model": PRIMARY_FUSION_MODEL,
    "primary_target_recall": PRIMARY_TARGET_RECALL,
    "calibration_method": CALIBRATION_METHOD,
    "backbone_name": BACKBONE_NAME,
    "image_size": IMAGE_SIZE,
    "epochs_image": EPOCHS_IMAGE,
    "epochs_cv": EPOCHS_CV,
    "exclude_benign_nodular_context_positives": EXCLUDE_BENIGN_NODULAR_CONTEXT_POSITIVES,
    "manual_exclude_uids": MANUAL_EXCLUDE_UIDS,
    "fusion_weight_grid_min": float(np.min(FUSION_WEIGHT_GRID)),
    "fusion_weight_grid_max": float(np.max(FUSION_WEIGHT_GRID)),
    "fusion_selection_mode": FUSION_SELECTION_MODE,
    "fusion_brier_lambda": FUSION_BRIER_LAMBDA,
    "fusion_recall_shortfall_penalty": FUSION_RECALL_SHORTFALL_PENALTY,
}

with open(OUTPUT_DIR / "manifest_v6_1.json", "w") as f:
    json.dump(manifest, f, indent=2)

readme_lines = [
    "IU-Xray multimodal V6.1",
    "======================",
    "",
    "Objetivo: evaluar si una fusión conservadora mejora o mantiene el desempeño de image_calibrated con sensibilidad alta.",
    "Input textual usado por el modelo: indication solamente.",
    "findings/impression se usan para construir/auditar etiquetas, no como entrada del modelo.",
    "",
    "Archivos clave:",
    "- config_v6_1.json",
    "- dataset_summary_v6_1.csv",
    "- label_audit_summary_v6_1.csv",
    "- main/summary_main_v6_1.csv",
    "- main/thresholds_main_v6_1.csv",
    "- main/predictions_test_main_v6_1.csv",
    "- main/predictions_test_uid_main_v6_1.csv",
    "- main/bootstrap_grouped_uid_ci_v6_1.csv",
    "- main/mcnemar_primary_threshold_v6_1.csv",
    "- cv_summary_v6_1.csv y cv_primary_aggregate_v6_1.csv si RUN_KFOLD_CV=True",
    "",
    "Nota: fusion_stack se conserva como análisis secundario; el resultado principal sigue siendo image_calibrated vs fusion_weighted conservador.",
]
with open(OUTPUT_DIR / "README_outputs_v6_1.txt", "w") as f:
    f.write("\n".join(readme_lines))

print(json.dumps(manifest, indent=2))
print("Manifiesto guardado en:", OUTPUT_DIR / "manifest_v6_1.json")


## 18. ZIP final de resultados

El ZIP final debe enviarse para comparar V6.1 contra V6. Los archivos más importantes serán:

- `dataset_summary_v6_1.csv`
- `benign_nodular_review_candidates_v6_1.csv`
- `main/summary_main_v6_1.csv`
- `main/fusion_weight_search_main_v6_1.csv`
- `main/fusion_weight_selected_main_v6_1.json`
- `main/bootstrap_grouped_uid_ci_v6_1.csv`
- `main/mcnemar_primary_threshold_v6_1.csv`
- `cv_primary_aggregate_v6_1.csv`


In [ ]:
zip_base = "/content/outputs_multimodal_iuxray_v6_1"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("ZIP creado en:", zip_path)
print("\nArchivos generados:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(OUTPUT_DIR)
        print("-", rel)
